<a href="https://colab.research.google.com/github/tav-rn/TavernTools/blob/main/Prover9_Mace4_Notebook_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction: Prover9/Mace4 with Python (in Colab)



This is a brief (and non-exhaustive) tutorial on using **Python** to run **Prover9/Mace4** inside a **Jupyter-style notebook** (Google Colab).

We assume some familiarity with:
- basic first-order logic (terms, equations, quantifiers),
- the general idea of automated theorem proving / finite model search,
- and elementary Python.

The guiding idea is:

> Prover9 and Mace4 do the grunt work.  
> Python (in a notebook) lets you be clever; helps you generate inputs, run many experiments, parse outputs, and visualize models.

---

## Prover9 and Mace4

[Prover9/Mace4](https://www.cs.unm.edu/~mccune/prover9/) are programs written by William McCune for reasoning about **first-order structures with equality**, including **functions** and **relations**.

Both tools take as input a block of text consisting of **assumptions** and (optionally) **goals**, written in the Prover9/Mace4 syntax (see the [syntax reference](https://www.cs.unm.edu/~mccune/prover9/manual/2009-11A/syntax.html)):
* An *ordinary symbol* is a (maximal) string made from the characters `a-z`, `A-Z`, `0-9`, `$`, and `_`.
* A *special symbol* is a (maximal) string made from the special characters: `{+-*/\^<>=`~?@&|!#';}`.
* A *quoted symbol* is any string enclosed in double quotes.
* The *empty list symbol* is `[]`. This is a special case.

For example, the group axioms can be expressed as:

```prover9
x * (y * z) = (x * y) * z.
x * 1 = x.
1 * x = x.
all x exists y (x * y = 1 & y * x = 1).
```

**Note.** Instead of the final sentence (existence of inverses), one can introduce a unary function (for inverse) and use identities such as  
`i(x) * x = 1.` and `x * i(x) = 1.`

## Prover9

**Prover9** attempts to derive a specified **goal** from the **assumptions**, if a proof exists.

A typical use is: supply some axioms (assumptions) and ask Prover9 to prove a target formula (goal). Prover9 outputs a derivation in a clause-based calculus (often as a refutation proof).

For example, if we take as Assumptions the group axioms above, and as Goal the sentence (expressing the identity element 1 is unique in any group):

> ```
>all y (x * y = y & y * x = y) -> x = 1.
>```

Prover9 halts with the following proof:

```
============================== PROOF =================================

% -------- Comments from original proof --------
% Proof 1 at 0.00 (+ 0.01) seconds.
% Length of proof is 7.
% Level of proof is 3.
% Maximum clause weight is 5.000.
% Given clauses 6.


2 (all x (y * x = x & x * y = x)) -> y = 1 # label(non_clause) # label(goal).  [goal].
5 x * 1 = x.  [assumption].
9 c1 * x = x.  [deny(2)].
11 1 != c1.  [deny(2)].
12 c1 != 1.  [copy(11),flip(a)].
17 c1 = 1.  [para(9(a,1),5(a,1)),flip(a)].
18 $F.  [resolve(17,a,12,a)].

============================== end of proof ==========================
```

* Prover9’s internal proofs are typically refutation proofs (proof by contradiction) in a clause-based calculus.
*	The output can be reformatted, expanded/contracted, and also used to generate hints to guide later runs.
*	Even when the proof by contradiction, you can often "read off" a direct argument.

## Mace4

**Mace4** searches for **finite models** of the assumptions, and if a goal is given it tries to find a **countermodel** (a model of the assumptions that refutes the goal). If no goal is provided, Mace4 searches for models of the assumptions.

Mace4 can also:
- search for countermodels of a goal,
- search for all models up to a given size,
- and remove isomorphic duplicates via tools such as `isofilter`.


For example, taking as Assumptions the group axioms above, and as Goal (commutativity):

>```
>x * y = y * x.
>```
   
Mace4 returns the smallest model, i.e., a nonabelian group of mininal cardinality:

```
interpretation( 6, [number = 1,seconds = 0], [
    function(*(_,_), [
        1,0,3,2,5,4,
        0,1,2,3,4,5,
        4,2,1,5,0,3,
        5,3,0,4,1,2,
        2,4,5,1,3,0,
        3,5,4,0,2,1]),
    function(c1, [0]),
    function(c2, [2]),
    function(f1(_), [0,1,2,4,3,5])]).
```
- the big list under *(_ , _) is the operation table (flattened),
-	f1 interprets the unary function symbol (here: an inverse map),
-	and c1, c2 are named constants (often used as witnesses when a goal fails).

For many more options and examples, see the Prover9/Mace4 documentation.

---




## Why Python?

Once you start doing many runs (varying axioms, changing parameters, enumerating structures, filtering outputs), it becomes convenient to let Python do the “glue work”:

- **Build inputs programmatically** (lists of strings, templates, generators),
- **Run Prover9/Mace4 from inside the notebook** and capture output,
- **Parse and clean outputs** (proofs, interpretations),
- **Turn Mace4 models into Python objects** you can compute with,
- **Analyze and visualize models** (tables, graphs, derived operations, checks of extra identities),
- **Post-process proofs** (reformatting, “humanizing” steps, extracting hints).
- **Reformat outputs** for other tools (Prover9 proofs into Lean syntax, or in LaTeX; Mace4 models into UACalc syntax, or Latex tables/tikz diagrams).


In other words: Prover9/Mace4 *find* proofs/models; Python helps you *work with* proofs/models.

---




## Why Jupyter-style notebooks (like Colab)?

A notebook is ideal for a tutorial because it mixes:
- explanations (Markdown),
- experiments (code),
- outputs (proofs, models, plots),
- reusable helper functions you can refine as you go.

Google Colab is especially convenient because it is free, runs in the browser, provides a ready-to-go Python environment, and makes it easy to share a runnable document via a link.
**However**, when inactive for long enough, **the session resets**, so you may lose data/values you've run.

For this reason, many people instead use a Jupyter notebook locally on their device.




## Colab basics

A Colab notebook is a document made of **cells**:
- **Text cells (Markdown)** for explanations, headings, links, equations, etc.
- **Code cells (Python)** that you can run interactively.

When you run code, it executes on a temporary remote machine (“runtime”). The notebook *remembers the state* (variables, imports, files) until you restart the runtime.


---

**Run the current cell**
- Click the **▶** button at the left of the cell, or
- Press **Option + Enter** (Mac) or **Alt + Enter** (Windows/Linux) to run and remain in current cell.
- Press **Shift + Enter** to run and move to next cell.
- Press **Option + Enter** (Mac) or **Alt + Enter** (Windows/Linux) to run and insert new cell below.

> You can run all cells via Menu: **Runtime → Run all**

Test:

In [ ]:
print("hello world")

hello world


**The notebook is not “one script”!**
If you run cells out of order, you can get confusing results (variables defined in a later cell, etc.).


# Load `provers` (must run first at startup)

To load the [GitHub repository of Peter Jipsen](https://github.com/jipsen/provers), you run the following cell.

**This must be run at the start of every session** (or reset) on Colab. Only once if on a local Jupyter notebook.


In [ ]:
!wget https://raw.githubusercontent.com/jipsen/provers/refs/heads/master/LoadProver9.py
%run LoadProver9.py

--2026-03-12 16:01:21--  https://raw.githubusercontent.com/jipsen/provers/refs/heads/master/LoadProver9.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2814 (2.7K) [text/plain]
Saving to: ‘LoadProver9.py.3’

LoadProver9.py.3    100%[===================>]   2.75K  --.-KB/s    in 0s      

2026-03-12 16:01:21 (36.7 MB/s) - ‘LoadProver9.py.3’ saved [2814/2814]

============================== Prover9 ===============================


## `prover9.py`

This loads all the functions in the file `prover9.py`, which can be found in this notebooks directory:


```
/content/provers/provers/prover9.py
```

Clicking the link above (or going through the directory on the left side bar, the folder icon) will open this file.

Here, you can inspect how these basic commands work. Why would you do this?
  - To understand what is going on if there is confusion.
  - To adapt, refine, redefine, these functions to your particular case. Simply copy/paste function definitions in a new cell and play.


In what follows, we will highlight and inspect some of these commands, and later on give examples on how to adapt them.

---

## Formula lists

An essential translation from Prover9/Mace4 syntax to that of `provers` is the use of lists of strings as formulas.

For example, recalling from above, the Prover9 syntax for the group axioms:
```prover9
x * (y * z) = (x * y) * z.
x * 1 = x.
1 * x = x.
all x exists y (x * y = 1 & y * x = 1).
```

Will be represented as a list of **strings** (with the `.`) removed.
 - simply copy/paste each line above (without the period `.`) inside quotes `" "` or `' '`

In [ ]:
forumla_list1 = ["x * (y * z) = (x * y) * z",
            "x * 1 = x",
            "1 * x = x",
            "all x exists y (x * y = 1 & y * x = 1)"
            ]

#or with involution
forumla_list2 = ["x * (y * z) = (x * y) * z",
            "x * 1 = x",
            "1 * x = x",
            "i(x) * x = 1",
            "x * i(x) = 1"
            ]
print("Group axioms (without involution in signature): ")
print(forumla_list1)
print("")
print("Group axioms (with involution in signature): ")
print(forumla_list2)

Group axioms (without involution in signature): 
['x * (y * z) = (x * y) * z', 'x * 1 = x', '1 * x = x', 'all x exists y (x * y = 1 & y * x = 1)']

Group axioms (with involution in signature): 
['x * (y * z) = (x * y) * z', 'x * 1 = x', '1 * x = x', 'i(x) * x = 1', 'x * i(x) = 1']


# The basic functions: `prover9()` and  `p9()`



## The `prover9()' function

The principle function used is called `prover9`, which invoke Prover9/Mace4 with lists of formulas and some (limited) options.

```
prover9(assume_list,
        goal_list,
        mace_seconds,
        prover_seconds,
        cardinality,
        options,
        one=False,
        noniso=True,
        info=False,
        params='')
```

**The main inputs:**

* `assume_list`: a list of Prover9 formulas that assumptions

* `goal_list`: a (possibly empty) list of Prover9 formulas that goals

* `mace_seconds`: number of seconds to run Mace4
> the default value is `2` (seconds)


* `prover_seconds`: number of seconds to run Prover9 (**only runs if** mace_seconds < 5)
> the default value is `60` seconds

* `cardinality`:
  - if `None`, search for 1 counter model staring from cardinality 2
  - if an integer `n` (>=2), search for all nonisomorphic models of
cardinality `n`.


Additional inputs:
* `options` -- list of prover9 options (default [], i.e. none)

* `one` -- if `True`, find only one model of given cardinality

* `noniso` -- if `True`, remove isomorphic copies

* `info` -- print input and output of prover9

---

**`prover9()` returns either a list of proofs (via Prover9), or a list of models (via Mace4)**

---

Example of a proof:

In [ ]:
# group axioms as a list, formula_list2 from above
group_ax = ["(x * y) * z = x * (y * z)",
         "x * 1 = x",
         "1 * x = x",
         "x * i(x) = 1",
         "i(x) * x = 1"
         ]
# a identitygoal satisfied by groups
goal = ["i(x * y) = i(y) * i(x)"]

# finds proof
prover9(group_ax, goal)

[
 Proof(syntax="Prover9", formula_list=[
 [1, 'i(x * y) = i(y) * i(x) # label(non_clause) # label(goal)', []],
 [2, 'x * y * z = x * (y * z)', []],
 [3, 'x * 1 = x', []],
 [4, '1 * x = x', []],
 [5, 'x * i(x) = 1', []],
 [6, 'i(x) * x = 1', []],
 [7, 'i(c1 * c2) != i(c2) * i(c1)', [1]],
 [8, '1 = x * (y * i(x * y))', [5, 2]],
 [9, 'x * (y * i(x * y)) = 1', [8]],
 [10, '1 * x = i(y) * (y * x)', [6, 2]],
 [11, 'x = i(y) * (y * x)', [4, 10]],
 [12, 'i(x) * (x * y) = y', [11]],
 [13, 'i(x) * 1 = y * i(x * y)', [9, 12]],
 [14, 'i(x) = y * i(x * y)', [3, 13]],
 [15, 'x * i(y * x) = i(y)', [14]],
 [16, 'i(x) * i(y) = i(y * x)', [15, 12]],
 [17, 'i(x * y) = i(y) * i(x)', [16]],
 [18, '$F', [17, 7]]])]

Example of countermodel:

In [ ]:
# group axioms as a list
group_ax = ["(x * y) * z = x * (y * z)",
         "x * 1 = x",
         "1 * x = x",
         "x * i(x) = 1",
         "i(x) * x = 1"
         ]

# the Goal (list)
comm = ["x * y = y *  x"]

# finds a counter-model of minimal cardinality
prover9(group_ax, comm)

[
 Model(cardinality = 6, index = 0,
 operations = {
 "*":[
 [1,0,3,2,5,4],
 [0,1,2,3,4,5],
 [4,2,1,5,0,3],
 [5,3,0,4,1,2],
 [2,4,5,1,3,0],
 [3,5,4,0,2,1]], "c1":0, "c2":2, "i":[0, 1, 2, 4, 3, 5]})]

Here is an example of finding a countermodel of a given cardinaltiy using `one = True`.

In [ ]:
# finds a counter-model of cardinality n = 10
prover9(group_ax, comm, cardinality = 10, one = True)

[
 Model(cardinality = 10, index = 0,
 operations = {
 "*":[
 [1,0,3,2,5,4,7,6,9,8],
 [0,1,2,3,4,5,6,7,8,9],
 [4,2,1,6,0,8,3,9,5,7],
 [5,3,0,7,1,9,2,8,4,6],
 [2,4,6,1,8,0,9,3,7,5],
 [3,5,7,0,9,1,8,2,6,4],
 [8,6,4,9,2,7,1,5,0,3],
 [9,7,5,8,3,6,0,4,1,2],
 [6,8,9,4,7,2,5,1,3,0],
 [7,9,8,5,6,3,4,0,2,1]], "c1":0, "c2":2, "i":[0, 1, 2, 4, 3, 5, 6, 8, 7, 9]})]

Example of all non-isomorphic models of a given cardinality (note here that the `goal_list` is the empty list `[]`):

In [ ]:
# group axioms as a list
group_ax = ["(x * y) * z = x * (y * z)",
         "x * 1 = x",
         "1 * x = x",
         "x * i(x) = 1",
         "i(x) * x = 1"
         ]
# commutativity
comm = ["x * y = y *  x"]

#abelian groups
abelian_group_ax = group_ax + comm

# finds all abelian groups of size 8
AbG = prover9(abelian_group_ax, [], mace_seconds = 60, cardinality = 8)

Number of nonisomorphic models of cardinality 8  is  3


Here, `AbG` is a list of 3 models.

In [ ]:
print(AbG)

[
Model(cardinality = 8, index = 0,
operations = {
"*":[
[1,0,3,2,5,4,7,6],
[0,1,2,3,4,5,6,7],
[3,2,1,0,6,7,4,5],
[2,3,0,1,7,6,5,4],
[5,4,6,7,1,0,2,3],
[4,5,7,6,0,1,3,2],
[7,6,4,5,2,3,1,0],
[6,7,5,4,3,2,0,1]], "i":[0, 1, 2, 3, 4, 5, 6, 7]}), 
Model(cardinality = 8, index = 1,
operations = {
"*":[
[1,0,3,2,5,4,7,6],
[0,1,2,3,4,5,6,7],
[3,2,1,0,6,7,4,5],
[2,3,0,1,7,6,5,4],
[5,4,6,7,0,1,3,2],
[4,5,7,6,1,0,2,3],
[7,6,4,5,3,2,0,1],
[6,7,5,4,2,3,1,0]], "i":[0, 1, 2, 3, 5, 4, 7, 6]}), 
Model(cardinality = 8, index = 2,
operations = {
"*":[
[1,0,3,2,6,7,4,5],
[0,1,2,3,4,5,6,7],
[3,2,0,1,5,6,7,4],
[2,3,1,0,7,4,5,6],
[6,4,5,7,3,1,2,0],
[7,5,6,4,1,2,0,3],
[4,6,7,5,2,0,3,1],
[5,7,4,6,0,3,1,2]], "i":[0, 1, 3, 2, 5, 4, 7, 6]})]


As with any list, `AbG[0]` is the initial/first entry, so `AbG[i-1]` is the i-th entry.

> **Note** the last entry of any list is obtained by using `-1` as entry.

In [ ]:
print(AbG[0])


Model(cardinality = 8, index = 0,
operations = {
"*":[
[1,0,3,2,5,4,7,6],
[0,1,2,3,4,5,6,7],
[3,2,1,0,6,7,4,5],
[2,3,0,1,7,6,5,4],
[5,4,6,7,1,0,2,3],
[4,5,7,6,0,1,3,2],
[7,6,4,5,2,3,1,0],
[6,7,5,4,3,2,0,1]], "i":[0, 1, 2, 3, 4, 5, 6, 7]})


In [ ]:
print(AbG[2])


Model(cardinality = 8, index = 2,
operations = {
"*":[
[1,0,3,2,6,7,4,5],
[0,1,2,3,4,5,6,7],
[3,2,0,1,5,6,7,4],
[2,3,1,0,7,4,5,6],
[6,4,5,7,3,1,2,0],
[7,5,6,4,1,2,0,3],
[4,6,7,5,2,0,3,1],
[5,7,4,6,0,3,1,2]], "i":[0, 1, 3, 2, 5, 4, 7, 6]})


In [ ]:
print(AbG[-1])


Model(cardinality = 8, index = 2,
operations = {
"*":[
[1,0,3,2,6,7,4,5],
[0,1,2,3,4,5,6,7],
[3,2,0,1,5,6,7,4],
[2,3,1,0,7,4,5,6],
[6,4,5,7,3,1,2,0],
[7,5,6,4,1,2,0,3],
[4,6,7,5,2,0,3,1],
[5,7,4,6,0,3,1,2]], "i":[0, 1, 3, 2, 5, 4, 7, 6]})


It is therefore easy to find all models of a given size with a simple loop:

In [ ]:
#All groups up to size 8

##We first create an empty list,
##and iteratively append to it the list of (noniso) models
##obtained from a run of Mace4 for a given cardinality
groups = []
for n in range(2,9):
  groups.append(prover9(group_ax,[], 60, cardinality = n))

Number of nonisomorphic models of cardinality 2  is  1
Number of nonisomorphic models of cardinality 3  is  1
Number of nonisomorphic models of cardinality 4  is  2
Number of nonisomorphic models of cardinality 5  is  1
Number of nonisomorphic models of cardinality 6  is  2
Number of nonisomorphic models of cardinality 7  is  1
Number of nonisomorphic models of cardinality 8  is  5


Now `groups` is a list of lists of models.

*   `groups[i]` is the list of all nonisomorphic models of cardinality i+2
*   `groups[i][j]` is the j-th model of cardinality i+2

However, this mismatch of cardinality and index is inconvient and confusing, so we "pad" (entries `0` and `1`) of such lists of lists of models of a given cardinality, call it `L`, so that
  - `L[0] = []`
  - `L[1] = [1]`
  - for n ≥ 2, `L[n]` are the models of cardinality n.

 Redoing the example above:  

In [ ]:
groups = [[],[1]]

for n in range(2,9):
  groups.append(prover9(group_ax,[], 60, cardinality = n))

print(len(groups))

Number of nonisomorphic models of cardinality 2  is  1
Number of nonisomorphic models of cardinality 3  is  1
Number of nonisomorphic models of cardinality 4  is  2
Number of nonisomorphic models of cardinality 5  is  1
Number of nonisomorphic models of cardinality 6  is  2
Number of nonisomorphic models of cardinality 7  is  1
Number of nonisomorphic models of cardinality 8  is  5
9


In [ ]:
groups[8][0]


Model(cardinality = 8, index = 0,
operations = {
"*":[
[1,0,3,2,5,4,7,6],
[0,1,2,3,4,5,6,7],
[3,2,1,0,6,7,4,5],
[2,3,0,1,7,6,5,4],
[5,4,6,7,1,0,2,3],
[4,5,7,6,0,1,3,2],
[7,6,4,5,2,3,1,0],
[6,7,5,4,3,2,0,1]], "i":[0, 1, 2, 3, 4, 5, 6, 7]})

##The `p9()` function

In fact, this behavior is implemented in an alternative function, `p9()`, that is usually the one used in practice/normal use. The inputs are basically the same.

```
p9(assume_list,
        goal_list,
        mace_seconds,
        prover_seconds,
        cardinality,
        options,
        noniso=True,
        info=False,
        params='')
```

The main difference is that `cardinality` accepts a new input:

* if  `cardinality = [n]` for some integer n ≥ 2, search for all nonisomorphic models up to, and including, cardinality n.


> *However, notice there is no longer the input* `one`, so when you want to find a single counterexample of a given cardinality you must use `prover9()`.

---

**`p9()` returns**
 - a **list of proofs** (via Prover9),

 or (via Mace4):
 - a **list of models** if `cardinality = n`, or
 - a **list of lists of models** if `cardinaltiy = [n]`.

---



Revisiting the example above.

In [ ]:
groups = p9(group_ax,[],cardinality = [8])

Number of nonisomorphic models of cardinality 2  is  1
Number of nonisomorphic models of cardinality 3  is  1
Number of nonisomorphic models of cardinality 4  is  2
Number of nonisomorphic models of cardinality 5  is  1
Number of nonisomorphic models of cardinality 6  is  2
Number of nonisomorphic models of cardinality 7  is  1
Number of nonisomorphic models of cardinality 8  is  5
Fine spectrum:  [1, 1, 1, 2, 1, 2, 1, 5]


In [ ]:
print("First model of cardinality 8:")
print(groups[8][0])

First model of cardinality 8:

Model(cardinality = 8, index = 0,
operations = {
"*":[
[1,0,3,2,5,4,7,6],
[0,1,2,3,4,5,6,7],
[3,2,1,0,6,7,4,5],
[2,3,0,1,7,6,5,4],
[5,4,6,7,1,0,2,3],
[4,5,7,6,0,1,3,2],
[7,6,4,5,2,3,1,0],
[6,7,5,4,3,2,0,1]], "i":[0, 1, 2, 3, 4, 5, 6, 7]})


---

Using Prover9 is identical.

In [ ]:
p9(group_ax, ["i(x * y) = i(y) * i(x)"])

[
 Proof(syntax="Prover9", formula_list=[
 [1, 'i(x * y) = i(y) * i(x) # label(non_clause) # label(goal)', []],
 [2, 'x * y * z = x * (y * z)', []],
 [3, 'x * 1 = x', []],
 [4, '1 * x = x', []],
 [5, 'x * i(x) = 1', []],
 [6, 'i(x) * x = 1', []],
 [7, 'i(c1 * c2) != i(c2) * i(c1)', [1]],
 [8, '1 = x * (y * i(x * y))', [5, 2]],
 [9, 'x * (y * i(x * y)) = 1', [8]],
 [10, '1 * x = i(y) * (y * x)', [6, 2]],
 [11, 'x = i(y) * (y * x)', [4, 10]],
 [12, 'i(x) * (x * y) = y', [11]],
 [13, 'i(x) * 1 = y * i(x * y)', [9, 12]],
 [14, 'i(x) = y * i(x * y)', [3, 13]],
 [15, 'x * i(y * x) = i(y)', [14]],
 [16, 'i(x) * i(y) = i(y * x)', [15, 12]],
 [17, 'i(x * y) = i(y) * i(x)', [16]],
 [18, '$F', [17, 7]]])]

# The `Proof` class

## `Proof` attributes

In [ ]:
# Remember, proofs are output as a list (as you may have more than one goal)
Ps = p9(group_ax, ["i(x * y) = i(y) * i(x)"],0,60)

# So lets look at its first entry
P = Ps[0]

# P is a proof
print("type(P) == Proof : ", type(P) == Proof)

#What P looks like
print("What the Proof P looks like:")
print(P)

type(P) == Proof :  True
What the Proof P looks like:

Proof(syntax="Prover9", formula_list=[
[1, 'i(x * y) = i(y) * i(x) # label(non_clause) # label(goal)', []],
[2, 'x * y * z = x * (y * z)', []],
[3, 'x * 1 = x', []],
[4, '1 * x = x', []],
[5, 'x * i(x) = 1', []],
[6, 'i(x) * x = 1', []],
[7, 'i(c1 * c2) != i(c2) * i(c1)', [1]],
[8, '1 = x * (y * i(x * y))', [5, 2]],
[9, 'x * (y * i(x * y)) = 1', [8]],
[10, '1 * x = i(y) * (y * x)', [6, 2]],
[11, 'x = i(y) * (y * x)', [4, 10]],
[12, 'i(x) * (x * y) = y', [11]],
[13, 'i(x) * 1 = y * i(x * y)', [9, 12]],
[14, 'i(x) = y * i(x * y)', [3, 13]],
[15, 'x * i(y * x) = i(y)', [14]],
[16, 'i(x) * i(y) = i(y * x)', [15, 12]],
[17, 'i(x * y) = i(y) * i(x)', [16]],
[18, '$F', [17, 7]]])


The `Proof` class is very simple, and has the following attributes:

  - `syntax`: a string that indicates what syntax is used for the formulas that represent the proof, e.g. "Prover9".
            
  - `proof` : a list of lists. Each list entry is a list of the form [formula, line_number, [references_to_preceding_lines]]. The references indicate which preceding lines are used in the derivation of the current line.

In [ ]:
#Attributes
print("The syntax:")
print(P.syntax)
print("")
print("The Proof P.proof, which is a a list of lines:")
print(P.proof)
print("")
print("   Each line is a triple, for example:")
print("")
print("The 10th line of proof P.proof[9]:")
print(P.proof[9])
print("   P.proof[9][0] is its line number (integer): ",P.proof[9][0])
print("   P.proof[9][1] is its formula (string): ",P.proof[9][1])
print("   P.proof[9][2] is its references (list): ", P.proof[9][2])

The syntax:
Prover9

The Proof P.proof, which is a a list of lines:
[[1, 'i(x * y) = i(y) * i(x) # label(non_clause) # label(goal)', []], [2, 'x * y * z = x * (y * z)', []], [3, 'x * 1 = x', []], [4, '1 * x = x', []], [5, 'x * i(x) = 1', []], [6, 'i(x) * x = 1', []], [7, 'i(c1 * c2) != i(c2) * i(c1)', [1]], [8, '1 = x * (y * i(x * y))', [5, 2]], [9, 'x * (y * i(x * y)) = 1', [8]], [10, '1 * x = i(y) * (y * x)', [6, 2]], [11, 'x = i(y) * (y * x)', [4, 10]], [12, 'i(x) * (x * y) = y', [11]], [13, 'i(x) * 1 = y * i(x * y)', [9, 12]], [14, 'i(x) = y * i(x * y)', [3, 13]], [15, 'x * i(y * x) = i(y)', [14]], [16, 'i(x) * i(y) = i(y * x)', [15, 12]], [17, 'i(x * y) = i(y) * i(x)', [16]], [18, '$F', [17, 7]]]

   Each line is a triple, for example:

The 10th line of proof P.proof[9]:
[10, '1 * x = i(y) * (y * x)', [6, 2]]
   P.proof[9][0] is its line number (integer):  10
   P.proof[9][1] is its formula (string):  1 * x = i(y) * (y * x)
   P.proof[9][2] is its references (list):  [6, 2]


---

## `p9latex()` and `p9lean()`

---
The function `p9latex()` allows one to (naively) translate a list of proofs into LaTeX for better readability (for some common symbols).
  - The first argument must be a list of proofs
  - The second argument `latex` is Boolean
    - if `False` (by default) displays the proof in tex
    - if `True` output the LaTeX code.

In [ ]:
p9latex(Ps)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
p9latex(Ps, latex = True)

'$1\\quad i(xy) = i(y)i(x)  \\quad(goal)\\quad []$\n\n$2\\quad xyz = x(yz)\\quad []$\n\n$3\\quad x1 = x\\quad []$\n\n$4\\quad 1x = x\\quad []$\n\n$5\\quad xi(x) = 1\\quad []$\n\n$6\\quad i(x)x = 1\\quad []$\n\n$7\\quad i(c_1c_2) \\ne  i(c_2)i(c_1)\\quad [1]$\n\n$8\\quad 1 = x(yi(xy))\\quad [5, 2]$\n\n$9\\quad x(yi(xy)) = 1\\quad [8]$\n\n$10\\quad 1x = i(y)(yx)\\quad [6, 2]$\n\n$11\\quad x = i(y)(yx)\\quad [4, 10]$\n\n$12\\quad i(x)(xy) = y\\quad [11]$\n\n$13\\quad i(x)1 = yi(xy)\\quad [9, 12]$\n\n$14\\quad i(x) = yi(xy)\\quad [3, 13]$\n\n$15\\quad xi(yx) = i(y)\\quad [14]$\n\n$16\\quad i(x)i(y) = i(yx)\\quad [15, 12]$\n\n$17\\quad i(xy) = i(y)i(x)\\quad [16]$\n\n$18\\quad F\\quad [17, 7]$'

---
The function `p9lean()` allows one to (naively) translate a list of proofs into Lean syntax.

In [ ]:
p9lean(Ps)

['1     i(x⬝y) = i(y)⬝i(x)    (goal)   []',
 '2     x⬝y⬝z = x⬝(y⬝z)   []',
 '3     x⬝1 = x   []',
 '4     1⬝x = x   []',
 '5     x⬝i(x) = 1   []',
 '6     i(x)⬝x = 1   []',
 '7     i(x₁⬝y₂) ≠ i(y₂)⬝i(x₁)   [1]',
 '8     1 = x⬝(y⬝i(x⬝y))   [5, 2]',
 '9     x⬝(y⬝i(x⬝y)) = 1   [8]',
 '10     1⬝x = i(y)⬝(y⬝x)   [6, 2]',
 '11     x = i(y)⬝(y⬝x)   [4, 10]',
 '12     i(x)⬝(x⬝y) = y   [11]',
 '13     i(x)⬝1 = y⬝i(x⬝y)   [9, 12]',
 '14     i(x) = y⬝i(x⬝y)   [3, 13]',
 '15     x⬝i(y⬝x) = i(y)   [14]',
 '16     i(x)⬝i(y) = i(y⬝x)   [15, 12]',
 '17     i(x⬝y) = i(y)⬝i(x)   [16]',
 '18     F   [17, 7]']

**Both `p9latex` and `p9lean` are only defined for a small list of specific symbols.** You can always append or change these by copying their source code into a cell, making adjustments, and then running that cell.

# The `Model` class

### `Model` attributes

In [ ]:
# Remember, Mace4 outputs a list models
Gs = p9(group_ax, [],5,0)

# So lets look at its first entry (there should only be one)
G = Gs[0]

#G is a Model
print("type(G) == Model : ", type(G) == Model)

print("")

#What G looks like
print("What the model G looks like:")
print(G)


type(G) == Model :  True

What the model G looks like:

Model(cardinality = 2, index = 0,
operations = {
"*":[
[1,0],
[0,1]], "i":[0, 1]})


The `Model` class has a bit more information.

  - `cardinality` : number of elements of the model's base set
  - `index` -- a natural number giving the position of the model
      in a list of models.
  - `operations` : a dictionary of operations on [0..cardinality-1].
      Entries are symbol:table pairs where symbol is a string
      that denotes the operation symbol, e.g. '+', and table is
      an n-dimensional array with entries from [0..cardinality-1].
      n >= 0 is the arity of the operation (not explicitly coded
      but can be computed from the table).
  - `relations` : a dictionary of relations on [0..cardinality-1].
      Entries are symbol:table pairs where symbol is a string
      that denotes the relation symbol, e.g. '<', and table is
      an n-dimensional array with entries from [0,1] (coding
      False/True). Alternatively the table can be an
      (n-2)-dimensional array with entries that are dictionaries
      with keys [0..cardinality-1] and values subsets of [0..cardinality-1],
      given as ordered lists.
      n >= 0 is the arity of the relation (not explicitly coded
      but can be computed from the table).

      ---
      Before we play with examples, let's generate some models with richer structure; in this case, we take bounded involutive lattices (along with the order relation):


In [ ]:
# lattice axioms
lattice_ax = ["x ^ (y ^ z) = (x ^ y) ^ z",
       "x ^ x = x",
       "x ^ y = y ^ x",
       "x v (y v z) = (x v y) v z",
       "x v x = x",
       "x v y = y v x",
       "x ^ (x v y) = x",
       "x v (x ^ y) = x"
       ]
# adding the natural order <=
lattice_w_order = lattice_ax + ["x <= y <-> x ^ y = x"]

# adding an antitone involution '
inv_lattice_ax = lattice_w_order + ["x'' = x", "x <= y -> y' <= x'"]

# add a largest element, a nullary constant T, for good measure.
bdd_inv_lattice_ax = inv_lattice_ax + ["x ^ T = x"]

# All bounded involutiove lattices up to cardinaltiy 6
biLat = p9(bdd_inv_lattice_ax, [],cardinality=[6])


Number of nonisomorphic models of cardinality 2  is  1
Number of nonisomorphic models of cardinality 3  is  1
Number of nonisomorphic models of cardinality 4  is  3
Number of nonisomorphic models of cardinality 5  is  4
Number of nonisomorphic models of cardinality 6  is  12
Fine spectrum:  [1, 1, 1, 3, 4, 12]


Let's pick a random model and inspect its attributes.

In [ ]:
#look at the model of index 8 (9th member) of cardinality 6
L = biLat[6][8]
L


Model(cardinality = 6, index = 8,
operations = {
"^":[
[0,1,2,3,4,5],
[1,1,1,1,1,1],
[2,1,2,1,1,2],
[3,1,1,3,4,4],
[4,1,1,4,4,4],
[5,1,2,4,4,5]], 
"v":[
[0,0,0,0,0,0],
[0,1,2,3,4,5],
[0,2,2,0,5,5],
[0,3,0,3,3,0],
[0,4,5,3,4,5],
[0,5,5,0,5,5]], "'":[1, 0, 3, 2, 5, 4], "T":0},
relations = {
"<=":[
[1,0,0,0,0,0],
[1,1,1,1,1,1],
[1,0,1,0,0,1],
[1,0,0,1,0,0],
[1,0,0,1,1,1],
[1,0,0,0,0,1]]})

In [ ]:
print("cardinality (integer):", L.cardinality)
print("")
print("index (integer or None):" , L.index)
print("")
print("operations (dictionary): ", L.operations)
print("")
print("key of operations sybmols (list of strings): ", L.operations.keys())
print("")
print("binary operation ^ (list of lists): ", L.operations["^"])
print("unary operation ' (list of integers): ", L.operations["'"])
print("nullary operation (constant) T (integer between 0,...,cardinality-1): ", L.operations["T"])
print("")
print("relations (dictionary): ", L.relations)
print("key of relation symbols (list of strings):", L.relations.keys())
print("relation <= (list of lists of Boolean values, here as 0/1): ", L.relations["<="])
print("")
print("")
print("The values for 3 v j, for j < cardinality: ", L.operations["v"][3])
print("The value for 3 v 2: ", L.operations["v"][3][2])


cardinality (integer): 6

index (integer or None): 8

operations (dictionary):  {'^': [[0, 1, 2, 3, 4, 5], [1, 1, 1, 1, 1, 1], [2, 1, 2, 1, 1, 2], [3, 1, 1, 3, 4, 4], [4, 1, 1, 4, 4, 4], [5, 1, 2, 4, 4, 5]], 'v': [[0, 0, 0, 0, 0, 0], [0, 1, 2, 3, 4, 5], [0, 2, 2, 0, 5, 5], [0, 3, 0, 3, 3, 0], [0, 4, 5, 3, 4, 5], [0, 5, 5, 0, 5, 5]], "'": [1, 0, 3, 2, 5, 4], 'T': 0}

key of operations sybmols (list of strings):  dict_keys(['^', 'v', "'", 'T'])

binary operation ^ (list of lists):  [[0, 1, 2, 3, 4, 5], [1, 1, 1, 1, 1, 1], [2, 1, 2, 1, 1, 2], [3, 1, 1, 3, 4, 4], [4, 1, 1, 4, 4, 4], [5, 1, 2, 4, 4, 5]]
unary operation ' (list of integers):  [1, 0, 3, 2, 5, 4]
nullary operation (constant) T (integer between 0,...,cardinality-1):  0

relations (dictionary):  {'<=': [[1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1], [1, 0, 1, 0, 0, 1], [1, 0, 0, 1, 0, 0], [1, 0, 0, 1, 1, 1], [1, 0, 0, 0, 0, 1]]}
key of relation symbols (list of strings): dict_keys(['<='])
relation <= (list of lists of Boolean values, h

## The `show()` command and ordered structures

The `show()` command converts a (list of) model(s), for some certain relations/operations, into a Hasse Diagram to be displayed.

In [ ]:
#Recall L is a lattice with connectives ^ and v
show(L,"v")

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 0--3 -->
 
 0--3 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 2--1 -->
 
 2--1 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 3--4 -->
 
 3--4 
 
 
<!-- 4--1 -->
 
 4--1 
 
 
<!-- 5--2 -->
 
 5--2 
 
 
<!-- 5--4 -->
 
 5--4

In [ ]:
# Recall biLat is a list of list of models
# Here are those models of cardinality 5
K = biLat[5]
show(K,"^")

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
 
 
        1<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
 
 
        2<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 3--4 -->
 
 3--4 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
 
 
        3<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 2--3 -->
 
 2--3 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--2 -->
 
 4--2

We can create a simple function that also shows a "class" of models.

In [ ]:
def Show(ls, ops = ["<= v"]):
  """
  ls -- a model, list of model, or list of lists of models
  ops -- a list of operations, as strings, to show
  """
  if type(ls)==Model: ls = [ls]
  if type(ls) == list:
    if type(ls[0])==Model:
      return show(ls, ops)
    elif type(ls[2])==list:
      ls = [ls[i] for i in range(len(ls)) if ls[i] != [] and ls[i] != [1]]
      if type(ls[0][0]) == Model:
        for i in range(len(ls)):
          if ls[i] != []:
            show(ls[i], ops)


In [ ]:
Show(biLat,"^")

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 1--0 -->
 
 1--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 2--0 -->
 
 2--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
 
 
        1<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
 
 
        2<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 2--3 -->
 
 2--3 
 
 
<!-- 3--0 -->
 
 3--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
 
 
        1<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
 
 
        2<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 3--4 -->
 
 3--4 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
 
 
        3<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 2--3 -->
 
 2--3 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--2 -->
 
 4--2

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 1--5 -->
 
 1--5 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 5--0 -->
 
 5--0 
 
 
 
 
        1<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 1--5 -->
 
 1--5 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 5--0 -->
 
 5--0 
 
 
 
 
        2<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 4--5 -->
 
 4--5 
 
 
<!-- 5--0 -->
 
 5--0 
 
 
 
 
        3<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 1--5 -->
 
 1--5 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 3--4 -->
 
 3--4 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 5--3 -->
 
 5--3 
 
 
 
 
        4<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 2--5 -->
 
 2--5 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 3--5 -->
 
 3--5 
 
 
<!-- 4--2 -->
 
 4--2 
 
 
<!-- 4--3 -->
 
 4--3 
 
 
<!-- 5--0 -->
 
 5--0 
 
 
 
 
        5<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 1--5 -->
 
 1--5 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 5--0 -->
 
 5--0 
 
 
 
 
        6<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1

In [ ]:
Show(biLat[6],"^")

In [ ]:
Show(biLat[6][0],"^")

### Output for tikz

The fumction `m4tikz()` outputs a tikz translation of the graphs, however there is a type in the `prover9.py` file. Here is a fix to that for it to run.

In [ ]:
def m4tikz(li,symbols="<= v", unaryRel=[]):
  # create a latex/tikz string for a list of digraphs
  st = ""
  for g in m4hasse(li,symbols="<= v", cols=unaryRel):
    st+=dot2tex.dot2tex(str(g), format='tikz', crop=True, figonly=True)+" \\qquad "
  return st

In [ ]:
m4tikz(biLat[6],"^")

'\n\\begin{tikzpicture}[>=latex,line join=bevel,]\n%%\n\\node (0) at (48.5bp,5.5bp) [draw=black,fill=white,circle] {0};\n  \\node (1) at (48.5bp,99.5bp) [draw=black,fill=white,circle] {1};\n  \\node (2) at (5.5bp,52.5bp) [draw=black,fill=white,circle] {2};\n  \\node (3) at (34.5bp,52.5bp) [draw=black,fill=white,circle] {3};\n  \\node (4) at (63.5bp,52.5bp) [draw=black,fill=white,circle] {4};\n  \\node (5) at (92.5bp,52.5bp) [draw=black,fill=white,circle] {5};\n  \\draw [] (1) ..controls (37.127bp,86.598bp) and (17.08bp,65.618bp)  .. (2);\n  \\draw [] (1) ..controls (44.423bp,85.395bp) and (38.567bp,66.572bp)  .. (3);\n  \\draw [] (1) ..controls (52.868bp,85.395bp) and (59.143bp,66.572bp)  .. (4);\n  \\draw [] (1) ..controls (60.218bp,86.515bp) and (81.059bp,65.202bp)  .. (5);\n  \\draw [] (2) ..controls (16.873bp,39.598bp) and (36.92bp,18.618bp)  .. (0);\n  \\draw [] (3) ..controls (38.577bp,38.395bp) and (44.433bp,19.572bp)  .. (0);\n  \\draw [] (4) ..controls (59.132bp,38.395bp) and 

## `diagram()`: Diagram of model

Lets start with an example class of models: `Lat` : lattices up to size 6

Some random lattice `L = Lat[6][3]`

In [ ]:
lattice_ax = ["x ^ (y ^ z) = (x ^ y) ^ z",
       "x ^ x = x",
       "x ^ y = y ^ x",
       "x v (y v z) = (x v y) v z",
       "x v x = x",
       "x v y = y v x",
       "x ^ (x v y) = x",
       "x v (x ^ y) = x"
       ]
Lat = p9(lattice_ax, [],60,0,cardinality=[6])

L = Lat[6][3]

print("L:")
print(L)
show(L,"v")

Number of nonisomorphic models of cardinality 2  is  1
Number of nonisomorphic models of cardinality 3  is  1
Number of nonisomorphic models of cardinality 4  is  2
Number of nonisomorphic models of cardinality 5  is  5
Number of nonisomorphic models of cardinality 6  is  15
Fine spectrum:  [1, 1, 1, 2, 5, 15]
L:

Model(cardinality = 6, index = 3,
operations = {
"^":[
[0,0,0,0,0,5],
[0,1,2,3,4,5],
[0,2,2,0,0,5],
[0,3,0,3,0,5],
[0,4,0,0,4,5],
[5,5,5,5,5,5]], 
"v":[
[0,1,2,3,4,0],
[1,1,1,1,1,1],
[2,1,2,1,1,2],
[3,1,1,3,1,3],
[4,1,1,1,4,4],
[0,1,2,3,4,5]]})


0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0

Sometimes its useful to feed (the diagram of) a model into the assumptions. This can be done by transforming the operation/relation tables into a list of identities.

The `diagram` (a `Model` class funtion) outputs this list of identites (i.e., the operation/relation tables written as formulas, line by line).


```
diagram(self,  # Model
        c,     # prefix string,
        s = 0, # Offset, default starting position is 0
        ):

```




Here, `diagram` makes it so elements are distinct. You can relax this condition by taking instead `positive_diagram`.

Below gives the exact diagram with the specified elements.

In [ ]:
L.diagram("")

['-(0=1)',
 '-(0=2)',
 '-(0=3)',
 '-(0=4)',
 '-(0=5)',
 '-(1=2)',
 '-(1=3)',
 '-(1=4)',
 '-(1=5)',
 '-(2=3)',
 '-(2=4)',
 '-(2=5)',
 '-(3=4)',
 '-(3=5)',
 '-(4=5)',
 '0 ^ 0 = 0',
 '0 ^ 1 = 0',
 '0 ^ 2 = 0',
 '0 ^ 3 = 0',
 '0 ^ 4 = 0',
 '0 ^ 5 = 5',
 '1 ^ 0 = 0',
 '1 ^ 1 = 1',
 '1 ^ 2 = 2',
 '1 ^ 3 = 3',
 '1 ^ 4 = 4',
 '1 ^ 5 = 5',
 '2 ^ 0 = 0',
 '2 ^ 1 = 2',
 '2 ^ 2 = 2',
 '2 ^ 3 = 0',
 '2 ^ 4 = 0',
 '2 ^ 5 = 5',
 '3 ^ 0 = 0',
 '3 ^ 1 = 3',
 '3 ^ 2 = 0',
 '3 ^ 3 = 3',
 '3 ^ 4 = 0',
 '3 ^ 5 = 5',
 '4 ^ 0 = 0',
 '4 ^ 1 = 4',
 '4 ^ 2 = 0',
 '4 ^ 3 = 0',
 '4 ^ 4 = 4',
 '4 ^ 5 = 5',
 '5 ^ 0 = 5',
 '5 ^ 1 = 5',
 '5 ^ 2 = 5',
 '5 ^ 3 = 5',
 '5 ^ 4 = 5',
 '5 ^ 5 = 5',
 '0 v 0 = 0',
 '0 v 1 = 1',
 '0 v 2 = 2',
 '0 v 3 = 3',
 '0 v 4 = 4',
 '0 v 5 = 0',
 '1 v 0 = 1',
 '1 v 1 = 1',
 '1 v 2 = 1',
 '1 v 3 = 1',
 '1 v 4 = 1',
 '1 v 5 = 1',
 '2 v 0 = 2',
 '2 v 1 = 1',
 '2 v 2 = 2',
 '2 v 3 = 1',
 '2 v 4 = 1',
 '2 v 5 = 2',
 '3 v 0 = 3',
 '3 v 1 = 1',
 '3 v 2 = 1',
 '3 v 3 = 3',
 '3 v 4 = 1',
 '3 v 5 =

> Note, the initial lines like `-(0=1)` are reduntant, as numerically assigned elements are always distinct.


In [ ]:
L.positive_diagram("")

['0 ^ 0 = 0',
 '0 ^ 1 = 0',
 '0 ^ 2 = 0',
 '0 ^ 3 = 0',
 '0 ^ 4 = 0',
 '0 ^ 5 = 5',
 '1 ^ 0 = 0',
 '1 ^ 1 = 1',
 '1 ^ 2 = 2',
 '1 ^ 3 = 3',
 '1 ^ 4 = 4',
 '1 ^ 5 = 5',
 '2 ^ 0 = 0',
 '2 ^ 1 = 2',
 '2 ^ 2 = 2',
 '2 ^ 3 = 0',
 '2 ^ 4 = 0',
 '2 ^ 5 = 5',
 '3 ^ 0 = 0',
 '3 ^ 1 = 3',
 '3 ^ 2 = 0',
 '3 ^ 3 = 3',
 '3 ^ 4 = 0',
 '3 ^ 5 = 5',
 '4 ^ 0 = 0',
 '4 ^ 1 = 4',
 '4 ^ 2 = 0',
 '4 ^ 3 = 0',
 '4 ^ 4 = 4',
 '4 ^ 5 = 5',
 '5 ^ 0 = 5',
 '5 ^ 1 = 5',
 '5 ^ 2 = 5',
 '5 ^ 3 = 5',
 '5 ^ 4 = 5',
 '5 ^ 5 = 5',
 '0 v 0 = 0',
 '0 v 1 = 1',
 '0 v 2 = 2',
 '0 v 3 = 3',
 '0 v 4 = 4',
 '0 v 5 = 0',
 '1 v 0 = 1',
 '1 v 1 = 1',
 '1 v 2 = 1',
 '1 v 3 = 1',
 '1 v 4 = 1',
 '1 v 5 = 1',
 '2 v 0 = 2',
 '2 v 1 = 1',
 '2 v 2 = 2',
 '2 v 3 = 1',
 '2 v 4 = 1',
 '2 v 5 = 2',
 '3 v 0 = 3',
 '3 v 1 = 1',
 '3 v 2 = 1',
 '3 v 3 = 3',
 '3 v 4 = 1',
 '3 v 5 = 3',
 '4 v 0 = 4',
 '4 v 1 = 1',
 '4 v 2 = 1',
 '4 v 3 = 1',
 '4 v 4 = 4',
 '4 v 5 = 4',
 '5 v 0 = 0',
 '5 v 1 = 1',
 '5 v 2 = 2',
 '5 v 3 = 3',
 '5 v 4 = 4',
 '5 v 

Or you can allow flexibility of the assigned elements by making the names variables (usually more useful), those element names take as prefix (to the numeral) the indicated `string` (here take to be `"a"`):

In [ ]:
L.diagram("a")

['-(a0=a1)',
 '-(a0=a2)',
 '-(a0=a3)',
 '-(a0=a4)',
 '-(a0=a5)',
 '-(a1=a2)',
 '-(a1=a3)',
 '-(a1=a4)',
 '-(a1=a5)',
 '-(a2=a3)',
 '-(a2=a4)',
 '-(a2=a5)',
 '-(a3=a4)',
 '-(a3=a5)',
 '-(a4=a5)',
 'a0 ^ a0 = a0',
 'a0 ^ a1 = a0',
 'a0 ^ a2 = a0',
 'a0 ^ a3 = a0',
 'a0 ^ a4 = a0',
 'a0 ^ a5 = a5',
 'a1 ^ a0 = a0',
 'a1 ^ a1 = a1',
 'a1 ^ a2 = a2',
 'a1 ^ a3 = a3',
 'a1 ^ a4 = a4',
 'a1 ^ a5 = a5',
 'a2 ^ a0 = a0',
 'a2 ^ a1 = a2',
 'a2 ^ a2 = a2',
 'a2 ^ a3 = a0',
 'a2 ^ a4 = a0',
 'a2 ^ a5 = a5',
 'a3 ^ a0 = a0',
 'a3 ^ a1 = a3',
 'a3 ^ a2 = a0',
 'a3 ^ a3 = a3',
 'a3 ^ a4 = a0',
 'a3 ^ a5 = a5',
 'a4 ^ a0 = a0',
 'a4 ^ a1 = a4',
 'a4 ^ a2 = a0',
 'a4 ^ a3 = a0',
 'a4 ^ a4 = a4',
 'a4 ^ a5 = a5',
 'a5 ^ a0 = a5',
 'a5 ^ a1 = a5',
 'a5 ^ a2 = a5',
 'a5 ^ a3 = a5',
 'a5 ^ a4 = a5',
 'a5 ^ a5 = a5',
 'a0 v a0 = a0',
 'a0 v a1 = a1',
 'a0 v a2 = a2',
 'a0 v a3 = a3',
 'a0 v a4 = a4',
 'a0 v a5 = a0',
 'a1 v a0 = a1',
 'a1 v a1 = a1',
 'a1 v a2 = a1',
 'a1 v a3 = a1',
 'a1 v a4 = a1',
 'a1 v

> Note, the initial lines like, e.g., `-(a0=a1)`, ensures that constant `a0` and the constant `a1` must be assigned to distinct elements from `0`, `...` ,`n-1` (`n` being the cardinality).


In [ ]:
L.positive_diagram("b")

['b0 ^ b0 = b0',
 'b0 ^ b1 = b0',
 'b0 ^ b2 = b0',
 'b0 ^ b3 = b0',
 'b0 ^ b4 = b0',
 'b0 ^ b5 = b5',
 'b1 ^ b0 = b0',
 'b1 ^ b1 = b1',
 'b1 ^ b2 = b2',
 'b1 ^ b3 = b3',
 'b1 ^ b4 = b4',
 'b1 ^ b5 = b5',
 'b2 ^ b0 = b0',
 'b2 ^ b1 = b2',
 'b2 ^ b2 = b2',
 'b2 ^ b3 = b0',
 'b2 ^ b4 = b0',
 'b2 ^ b5 = b5',
 'b3 ^ b0 = b0',
 'b3 ^ b1 = b3',
 'b3 ^ b2 = b0',
 'b3 ^ b3 = b3',
 'b3 ^ b4 = b0',
 'b3 ^ b5 = b5',
 'b4 ^ b0 = b0',
 'b4 ^ b1 = b4',
 'b4 ^ b2 = b0',
 'b4 ^ b3 = b0',
 'b4 ^ b4 = b4',
 'b4 ^ b5 = b5',
 'b5 ^ b0 = b5',
 'b5 ^ b1 = b5',
 'b5 ^ b2 = b5',
 'b5 ^ b3 = b5',
 'b5 ^ b4 = b5',
 'b5 ^ b5 = b5',
 'b0 v b0 = b0',
 'b0 v b1 = b1',
 'b0 v b2 = b2',
 'b0 v b3 = b3',
 'b0 v b4 = b4',
 'b0 v b5 = b0',
 'b1 v b0 = b1',
 'b1 v b1 = b1',
 'b1 v b2 = b1',
 'b1 v b3 = b1',
 'b1 v b4 = b1',
 'b1 v b5 = b1',
 'b2 v b0 = b2',
 'b2 v b1 = b1',
 'b2 v b2 = b2',
 'b2 v b3 = b1',
 'b2 v b4 = b1',
 'b2 v b5 = b2',
 'b3 v b0 = b3',
 'b3 v b1 = b1',
 'b3 v b2 = b1',
 'b3 v b3 = b3',
 'b3 v b4 = b1

> Here, constants (such as `b0` and `b1`) may be assigned to the same element.

---

### Using the diagrams:

In [ ]:
Ld = p9(L.diagram("a"),[],60,0,cardinality=[L.cardinality])

No model of cardinality 2
No model of cardinality 3
No model of cardinality 4
No model of cardinality 5
Number of nonisomorphic models of cardinality 6  is  1
Fine spectrum:  [1, 0, 0, 0, 0, 1]


---
Recall, the model `L` is a member of the class `Lat` of models satisying the formula list:

In [ ]:
lattice_ax = ['x ^ (y ^ z) = (x ^ y) ^ z',
              'x ^ x = x',
              'x ^ y = y ^ x',
              'x v (y v z) = (x v y) v z',
              'x v x = x',
              'x v y = y v x',
              'x ^ (x v y) = x',
              'x v (x ^ y) = x']

One thing we can do is find all models satisfying `lattice_ax`, up to a given cardinality, for which `L` is a substructure (i.e., that `L` embeds into).

In [ ]:
ex = lattice_ax + L.diagram("a")
L_ex = p9(ex,[],60,cardinality=[L.cardinality + 2])

Show(L_ex,"v")

No model of cardinality 2
No model of cardinality 3
No model of cardinality 4
No model of cardinality 5
Number of nonisomorphic models of cardinality 6  is  1
Number of nonisomorphic models of cardinality 7  is  6
Number of nonisomorphic models of cardinality 8  is  37
Fine spectrum:  [1, 0, 0, 0, 0, 1, 6, 37]


0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 1--6 -->
 
 1--6 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6--0 -->
 
 6--0 
 
 
 
 
        1<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 1--6 -->
 
 1--6 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6--4 -->
 
 6--4 
 
 
 
 
        2<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 1--6 -->
 
 1--6 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6--5 -->
 
 6--5 
 
 
 
 
        3<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 4--6 -->
 
 4--6 
 
 
<!-- 6--5 -->
 
 6--5 
 
 
 
 
        4<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 0--6 -->
 
 0--6 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 6--5 -->
 
 6--5 
 
 
 
 
        5<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 6--1 -->
 
 6--1

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 1--6 -->
 
 1--6 
 
 
<!-- 7 -->
 
 7 
 
 7 
 
<!-- 1--7 -->
 
 1--7 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6--0 -->
 
 6--0 
 
 
<!-- 7--0 -->
 
 7--0 
 
 
 
 
        1<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 7 -->
 
 7 
 
 7 
 
<!-- 1--7 -->
 
 1--7 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 6--0 -->
 
 6--0 
 
 
<!-- 7--6 -->
 
 7--6 
 
 
 
 
        2<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 1--6 -->
 
 1--6 
 
 
<!-- 7 -->
 
 7 
 
 7 
 
<!-- 1--7 -->
 
 1--7 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6--0 -->
 
 6--0 
 
 
<!-- 7--5 -->
 
 7--5 
 
 
 
 
        3<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 1--6 -->
 
 1--6 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 6--0 -->
 
 6--0 
 
 
<!-- 7 -->
 
 7 
 
 7 
 
<!-- 6--7 -->
 
 6--7 
 
 
<!-- 7--5 -->
 
 7--5 
 
 
 
 
        4<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 7 -->
 
 7 
 
 7 
 
<!-- 0--7 -->
 
 0--7 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 1--6 -->
 
 1--6 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 6--0 -->
 
 6--0 
 
 
<!-- 7--5 -->
 
 7--5 
 
 
 
 
        5<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -

### `inS()`, checking substructures

The function `inS()` checks the model is a subalgebra of another.


```
inS(self,       #Model A
    B,          #Model or cardinalty > A
    info=False
    )
```



For instance, find the collection of (non-isomorphic copies of) `L`'s substrucutres

In [ ]:
S_L = [[A for A in C if A.inS(L)] for C in Lat[2:L.cardinality+1]]

Show(S_L,"v")

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 1--0 -->
 
 1--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 2--0 -->
 
 2--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
 
 
        1<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--2 -->
 
 3--2

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0 
 
 
 
 
        1<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 0--4 -->
 
 0--4 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0

### `inH()` and homomorphic images

The function `inH()` checks the model is a homomorphic image of another.


```
inH(self,       #Model A
    B,          #Model or cardinalty > A
    info=False
    )
```



For instance, the collection of `L`'s homomorphic images (up to isomorphism).

In [ ]:
H_L = [[A for A in C if A.inH(L)] for C in Lat[2:L.cardinality+1]]

Show(H_L,"v")

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 1--0 -->
 
 1--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 0--5 -->
 
 0--5 
 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 1--4 -->
 
 1--4 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4--0 -->
 
 4--0

### `product()`, direct products

The function `product()` checks the model is a subalgebra of another.


```
product(self,       #Model A
    B,              #Model B
    info=False
    )
```



In [ ]:
#Product of two lattice
A = Lat[3][0]
B = Lat[4][0]
AB = A.product(B)
show(A,"v")
show(B,"v")
show(AB,"v")

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 2--0 -->
 
 2--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0

0<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 2.43.0 (0)
 -->
<!-- Title: %3 Pages: 1 -->
 
 
 %3 
 
<!-- 0 -->
 
 0 
 
 0 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 2 
 
<!-- 1--2 -->
 
 1--2 
 
 
<!-- 3 -->
 
 3 
 
 3 
 
<!-- 1--3 -->
 
 1--3 
 
 
<!-- 2--0 -->
 
 2--0 
 
 
<!-- 3--0 -->
 
 3--0 
 
 
<!-- 4 -->
 
 4 
 
 4 
 
<!-- 8 -->
 
 8 
 
 8 
 
<!-- 4--8 -->
 
 4--8 
 
 
<!-- 5 -->
 
 5 
 
 5 
 
<!-- 6 -->
 
 6 
 
 6 
 
<!-- 5--6 -->
 
 5--6 
 
 
<!-- 7 -->
 
 7 
 
 7 
 
<!-- 5--7 -->
 
 5--7 
 
 
<!-- 9 -->
 
 9 
 
 9 
 
<!-- 5--9 -->
 
 5--9 
 
 
<!-- 6--4 -->
 
 6--4 
 
 
<!-- 10 -->
 
 10 
 
 10 
 
<!-- 6--10 -->
 
 6--10 
 
 
<!-- 7--4 -->
 
 7--4 
 
 
<!-- 11 -->
 
 11 
 
 11 
 
<!-- 7--11 -->
 
 7--11 
 
 
<!-- 8--0 -->
 
 8--0 
 
 
<!-- 9--1 -->
 
 9--1 
 
 
<!-- 9--10 -->
 
 9--10 
 
 
<!-- 9--11 -->
 
 9--11 
 
 
<!-- 10--2 -->
 
 10--2 
 
 
<!-- 10--8 -->
 
 10--8 
 
 
<!-- 11--3 -->
 
 11--3 
 
 
<!-- 11--8 -->
 
 11--8

# Congruences and Subdirectly Irreducibles

In [ ]:
import contextlib #these are used for removing unwanted printing in calls
import os


def hide_print(command):
  with open(os.devnull, 'w') as devnull: #This is to removing printing from Con
    with contextlib.redirect_stdout(devnull):
      out = command
  return out  # Calls your function without printing to the console

def Con_lat(alg):
  with open(os.devnull, 'w') as devnull: #This is to removing printing from Con
    with contextlib.redirect_stdout(devnull):
          cL = Con(alg)  # Calls your function without printing to the console
  return cL

def isSI(algebra,return_monolith = False):
  def leq(p,q):
    return all(any(x<=y for y in q) for x in p)

  def nontriv(p):
    return any(len(x)>1 for x in p)

  def has_monolith(cL,return_mon): #cL should be a congruence lattice, so input Con(A) for an algebra A
    for p in cL:
      if all(nontriv(p) and (not nontriv(q) or leq(p,q)) for q in cL):
        if return_mon:
          return sorted([list(y) for y in p])
          # return p
        else: return True
        # return p
        # return True
    return False

  with open(os.devnull, 'w') as devnull: #This is to removing printing from Con
    with contextlib.redirect_stdout(devnull):
          cL = Con(algebra)  # Calls your function without printing to the console
  return has_monolith(cL,return_monolith)





#######find subdirectly irreducibles
def find_SI(models):
  if type(models)==Model: models=[models] #if models is not a class of models, but a single algebra A, redfine models as the class [A]
  if type(models[0]) == Model: #if models is a class of algebras
    SI = [A for A in models if isSI(A)]
  else: #if models is a class of classes of algebras
    SI = [[],[]] #Set SIs[0] empty for indexing, and SIs[1] empty for convention that trivial algebras aren't SI
    for i in range(2,len(models)):
      SI.append([A for A in models[i] if isSI(A)])
  return SI

# Hasse Diagrams


In [ ]:
connectives_dict ={
    "^" : ["m","l"],
    "v" : ["a","r"],
    "*" : ["m","l"],
    "+" : ["a","r"]
}
def additive(op):
  if connectives_dict[op][0] == "a":
    return True
  return False
def edgerev(p,rev=False):
    if rev: return (p[1],p[0])
    return (p[0],p[1])



def bin_ops(model):
  def is_bin_op(arr):
    if type(arr) == list:
      a = arr[0]
      if type(a) == list:
        e = a[0]
        if type(e) == int:
          return True
        else:
          return False
      else:
        return False
    else:
      return False
  return [s for s in model.relations.keys() if is_bin_op(model.relations[s])] + [s for s in model.operations.keys() if is_bin_op(model.operations[s])]

def poset2model_t(A):
    if len(A)==0: raise Error("Can't show Hasse diagram of an empty set")
    k = sorted(A, key = len)
    S = range(len(A))
    if all(type(x)==frozenset for x in k[0]):
      U = union(k[0])
      if all(all(type(y)==frozenset for y in x) and union(x)==U for x in k[1:]): #assume K is a set of partitions
        li = [[all(any(x<=y for y in k[j]) for x in k[i]) for i in S] for j in S]
      else: li = [[k[i]<=k[j] for i in S] for j in S]
    else: li = [[k[i]<=k[j] for i in S] for j in S]
    return Model(cardinality=len(k),relations={"<=":li})



import networkx as nx
from graphviz import Graph
from IPython.display import display_html
def hasse_diagram_t(op,op_s,rel,dual,color_n=[],elements=[],shape_n=[]):
    A = range(len(op))
    if elements == []:
      elements = [str(i) for i in A]
    G = nx.DiGraph()
    # def edgerev(p,rev=False):
    #   if rev: return (p[1],p[0])
    #   return (p[0],p[1])
    # op = op[0]
    if rel:
        G.add_edges_from([(x,y) for x in A for y in A if (op[x][y] if dual else op[y][x]) and x!=y])
        # G.add_edges_from([(x,y) for x in A for y in A if (op[y][x] if dual else op[x][y]) and x!=y])
    else:
        G.add_edges_from([edgerev([x,y], not additive(op_s) if dual else additive(op_s)) for x in A for y in A if op[x][y] == (x if dual else y) and x!=y])
    try:
        G = nx.algorithms.dag.transitive_reduction(G)
    except:
        pass
    P = Graph()
    P.attr('node', shape='circle', width='.3', height='.3', fixedsize='true', fontsize='10')
    # P.attr('node', shape='circle',   fontsize='10')
    for x in A: P.node(str(x), shape = shape_n[x],color= color_n[x],label = elements[x])
    # for x in A: P.node(str(x), color='black',label = elements[x])
    # for x in A: P.node(str(x), color='black')
    P.edges([(str(x[0]),str(x[1])) for x in G.edges])
    return P

def m4diag_t(li,symbols="<= v", color_node="",shape_node = "",name_func = "",show_tables = []):
  # use graphviz to display a mace4 structure as a diagram
  # symbols is a list of binary symbols that define a poset or graph
  # unaryRel is a unary relation symbol that is displayed by red nodes
  i = -1
  sy = symbols.split(" ")
  #print(symbols,"***",sy)
  st = ""
  for x in li:
    if name_func == "": elements =[]
    elif type(name_func) == dict or type(name_func) == list: elements = name_func
    else:
      elements = name_func(x)
    i+=1
    st+=str(i)
    if type(color_node) == str:
      uR = x.relations[color_node] if color_node!="" else ["black"]*x.cardinality
    elif type(color_node) == list  or type(color_node) == dict or type(color_node) == set :
      if len(color_node) == x.cardinality:
        uR = color_node
      else: uR = ["black"]*x.cardinality
    else:
      # print("Either unaryRel is not a relations, or is a list of wrong length")
      uR = color_node(x)

    if type(shape_node) == str:
      iD = ["square" if x.operations[shape_node][i][i] == i else "circle" for i in range(x.cardinality)] if shape_node!="" else ["circle"]*x.cardinality
    elif type(shape_node) == list  or type(shape_node) == dict or type(shape_node) == set :
      if len(color_node) == x.cardinality:
        iD = color_node
      else: iD = ["circle"]*x.cardinality
    else:
      # print("Either unaryRel is not a relations, or is a list of wrong length")
      iD = shape_node(x)
    for s in sy:
            t = s[:-1] if s[-1] == 'd' else s #(s[-1]=='d' or s[-1]=='r')else s
            if t in x.operations.keys():
                st+=hasse_diagram_t(x.operations[t],t,False,s[-1]=='d',uR,elements,iD)._repr_image_svg_xml()+"&nbsp; &nbsp; &nbsp; "
            elif t in x.relations.keys():
                st+=hasse_diagram_t(x.relations[t],t, True, s[-1]=='d',uR,elements,iD)._repr_image_svg_xml()+"&nbsp; &nbsp; &nbsp; "
    st+=" &nbsp; "
  display_html(st,raw=True)


def show_t(K,ops=[],names = "",tables = [],color_node = "", shape_node = ""): # show a list of Mace4 models using graphviz or show a set of subsets or partitions
  if type(K)==Model: K=[K]
  if type(K)==str: K = []
  if type(K)==list and len(K)>0 and type(K[0])==Model:
    def num_elms(m):
      return {i : [str(i),"",""] for i in range(m.cardinality)}
    if names == "": names = num_elms
    def nh(m):
      if (type(names) == list) or (type(names) == dict): return name_format(names)
      else: return name_format(names(m))
    def nt(m):
      if (type(names) == list) or (type(names) == dict): return name_format(names, "tex")
      else: return name_format(names(m),"tex")
    if type(ops) == str:
      st = ops
    else:
      if ops==[]:
        ops = bin_ops(K[0])
      ops=[x.strip() for x in ops]
      st=" ".join(ops)
    if tables:
      for A in K:
        m4diag_t([A],st, color_node, shape_node, name_func = nh, show_tables = tables)
        # m4diag_t([A],st,name_func = names, show_tables = tables)
        for op in tables:
          op_tables_tex(A,[op],nt(A))
    else:
      m4diag_t(K,st, color_node, shape_node, name_func = nh, show_tables = tables)
  elif type(K) == list and len(K)> 2 and type(K[2])==list: # and type(K[2][0]) == Model:
    for i in range(len(K)):
      if len(K[i])>0 and type(K[i][0]) == Model:
        print("-"*20 + f"Cardinality {K[i][0].cardinality}"+ "-"*20)
        show_t(K[i],ops,names,tables, color_node, shape_node)
        #print("-"*20 + f"Cardinality {K[i][0].cardinality}"+ "-"*20)
        #show_t(K[i],ops,names)
  elif type(K)==frozenset: m4diag_t([poset2model_t(K)],"<=d",name_func = names)
  elif type(K)==list: m4diag_t([poset2model(x) for x in K],name_func = names)



In [ ]:
def name_format(names, format = "html"):
  out = []
  for i in range(len(names)):
    let = names[i][0]
    # sub = names[i][1] if names[0][1] else ""
    # sup = names[i][2] if names[0][2] else ""
    sub = "" if len(names[i]) <= 1 else names[i][1]
    sup = "" if len(names[i]) <= 2 else names[i][2]
    if let == "epsilon": let = "&"+let+";" if format == "html" else r'\epsilon'
    elif let == "Delta": let = "&"+let+";" if format == "html" else r'\Delta'
    elif let == "nabla": let = "&"+let+";" if format == "html" else r'\nabla'
    elif let == "top": let = "T" if format == "html" else r'\top'
    if not sub == "": sub = "<sub>"+sub+"</sub>" if format == "html" else "_{"+sub+"}"
    if not sup == "": sup = "<sup>"+sup+"</sup>" if format == "html" else "^{"+sup+"}"
    n = f"<{let}{sub}{sup}>" if format == "html" else let + sub + sup

    out.append(n)
  return out

import string
def block_letters(n):
    alphabet = string.ascii_lowercase
    result = []
    # Loop to handle cases when n exceeds the alphabet length
    for i in range(n):
        letter = alphabet[i % 26]
        subscript = i // 26  # Determines if a subscript is needed
        if subscript > 0:
            #result.append(f"{letter}_{{{subscript}}}")
            result.append(letter * (subscript+1))
        else:
            result.append(letter)
    return result

#name dictionaries

def inv_dict(A):
  N = A.cardinality
  N = range(N)
  n = A.operations["'"]
  names = {}
  E = [e for e in N if n[e] == e]
  k = 0
  for e in E:
    if e == 1:
      names.update({1 : ["1","",""]})
    else:
      k += 1
      e_s = str(k) if len(E) > 1 else ""
      names.update({e : ['epsilon',str(k),""]})
  M = [e for e in N if n[e] != e]
  letters = block_letters(len(M))
  redun = []
  l = 0
  for b in M:
    if not(b in redun):
      redun += [n[b],b]
      if b == 1 or b == n[1]:
        names.update({1 : ["1","",""]})
        names.update({n[1] : ["0","",""]})
      else:
        letter = letters[l]
        names.update({b : [letter,"",""]})
        names.update({n[b] : [letter+"'","",""]})
        l += 1
  return names

def let_dict(A):
  N = A.cardinality
  N = range(N)
  # n = A.operations["'"]
  names = {1 : ["1","",""], 0 : ["0","",""]}
  M = [e for e in N if e != 1 and e != 0]
  # E = [e for e in N if n[e] == e]
  # k = 0
  # for e in E:
  #   if e == 1:
  #     names.update({1 : ["1",""]})
  #   else:
  #     k += 1
  #     e_s = str(k) if len(E) > 1 else ""
  #     names.update({e : ['epsilon',str(k)]})
  # M = [e for e in N if n[e] != e]
  letters = block_letters(len(M))
  redun = []
  l = 0
  for b in M:
    letter = letters[l]
    names.update({b : [letter,"",""]})
    l += 1
  return names

def idem_rig_dict(A, join = "+"):
  N = range(A.cardinality)
  j = A.operations[join]
  names = {1 : ["1","",""], 0 : ["0","",""]}
  M = [e for e in N if e != 1 and e != 0]
  # for e in M:
  #   if all(j[e][x] == e for x in N):
  #     names.update({e : ["top",""]})
  #     top = e
  letters = block_letters(len(M))
  l = 0
  for b in M:
    if all(j[b][x] == b for x in N):
      names.update({b : ["top","",""]})
    else:
      names.update({b : [letters[l],"",""]})
      l += 1
  return names

In [ ]:
#shape/color dicts
def monolith_class_dict(model):
  color_list = ["red",        #0
                "green",      #1
                "blue",       #2
                "purple",     #3
                "pink",       #4
                "brown",      #5
                "cyan",       #6
                "gray",       #7
                "lightblue"   #8
                ]
  mon = isSI(model,True)
  if mon:
    non_triv_eqvc = [eqvc for eqvc in mon if len(eqvc)>1]
    out_dic = {i : "black" for i in range(model.cardinality) if not any(i in eqvc for eqvc in non_triv_eqvc)}
    for j,eqvc in enumerate(non_triv_eqvc):
      for i in eqvc:
        out_dic.update({i : color_list[j]})
  else:
    out_dic = {i : "black" for i in range(model.cardinality)}
  return out_dic


def idrig_shape_dict(model, times = "*", one = 1 , join = "+", bot = 0):
  m = model.operations[times]
  j = model.operations[join]
  N = model.cardinality
  top = [x for x in range(N) if all(j[x][y] == x for y in range(N))][0]
  out = {}
  for x in range(N):
    if m[x][x] == x:
      out.update({x : "doublecircle"})
      # out.update({x : "invtriangle"})
    elif m[x][x] == top:
      out.update({x : "triangle"})
    elif m[x][x] == bot:
      out.update({x : "invtriangle"})
    elif m[x][x] == one:
      out.update({x : "diamond"})
    else:
      out.update({x : "circle"})
  return out


def mult_rig_dict(A, join = "+",times = "*"):
  n = A.cardinality
  N = range(n)
  j = A.operations[join]
  m = A.operations[times]
  names = {1 : ["1","",""], 0 : ["0","",""]}
  top = [x for x in N if all(j[x][y] == x for y in N)][0]
  if top != 1:
    names.update({top : ["top","",""]})
  M = [e for e in N if e != 1 and e != 0 and e != top]
  def Sg(x):
    x_p = x
    out = [x_p]
    for i in N:
      x_p = m[x_p][x]
      if x_p in out:
        return out
      else: out.append(x_p)
    return sorted(list(set(out)))
  max_Sg = []
  for x in M:
    S_x = Sg(x)
    test_x = [X for X in max_Sg if set(X).issubset(set(S_x))]
    if test_x == []:
      max_Sg.append(S_x)
    else:
      max_Sg = [S_x if set(X).issubset(set(S_x)) else X for X in max_Sg]
  letters = block_letters(len(max_Sg))
  for i,X in enumerate(max_Sg):
    for x in X:
      if not (x in names):
        sup = f"{X.index(x) + 1}" if X.index(x) != 0 else ""
        names.update({x : [letters[i],"",sup]})
  return names

# Axioms for structures


In [ ]:
axiom_dict = {
    "associative" : "x * (y * z) = (x * y) * z",
    "commutative" : "x * y = y * x",
    "idempotent" : "x * x = x",
    "left regular" : "(x * y) * x = x * y",
    "right regular" : "x * (y * x) = y * x",
    "right unit" : "x * 1 = x",
    "left unit" : "1 * x = x",
    "left zero" : "0 * x = 0",
    "right zero" : "x * 0 = 0",
    "left distributive": "x * (y + z) = (x * y) + (x * z)",
    "right distributive": "(x + y) * z = (x * z) + (y * z)",
    "locally commutative": "(x*1')' * (y*1')' = (y*1')' * (x*1')'",
    "left absorbing": "x * (x + y) = x",
    "right absorbing": "(x + y) * y = y",
    "absorbing left":"(x + y) * x = x",
    "absorbing right":"y * (x + y) = y",
    "involutive" : "x'' = x",
    "DeMorgan" : "x + y = (x' * y')'",
    "subclassical" : "1' = 0",
    "left locally complemented" : "x * x' = x * 1'",
    "left divisible" : "x * (x *  y')' = x * y",
    "right orthodistributive" : "(x' * x)' * y = ((x * y)' * (x' * y)')'",
    "left paracommutative" : "(x * y') * x = x * y",
    "left paradistributive" : "(x * y)' * z = (x' * z)' * (x' * (y * z))'",
    "left coherent" : "((x * x')' * y)' = (x' * x)' * y'",
    "Kleene" : "(x * x') * (y * y')' = x * x' "
}


def split_words(s: str) -> list[str]:
    return [part.strip() for part in s.split(',') if part.strip()]

def axioms(props):
  props = split_words(props)
  ax = []
  for prop in props:
    ax += [axiom_dict[prop]]
  return ax

def monoid(props = "", op = "*", unit = "1"):
  ax = "associative, right unit, left unit"
  if not props == "":
    ax = ax + ", " + props
  mon = [e.replace("*",op).replace("1",unit) for e in axioms(ax)]
  return mon



def iuband(op = "*", inv = "'", unit = "1"):
  ax = monoid("idempotent",op,unit) + axioms('involutive')

def McCarthy(op = "*", inv = "'", unit = "1"):
  n = axioms('involutive, subclassical, left zero, left divisible, right orthodistributive, locally commutative')
  ax = monoid("idempotent",op,unit) + [e.replace("'",inv).replace("*",op).replace("1",unit) for e in n]
  # for e in ax:
  #   e.replace("'",inv)
  #   e.replace("*",op)
  return ax


def unitalSL(op = "v", unit = "0"):
  ax = monoid("idempotent, commutative",op,unit)
  return ax

def band(op = "*", props = "", unit = False):
  #base = "idempotent, associative"
  ax_ls = ["associative","idempotent"] + split_words(props)
  if "unital" in ax_ls:
    ax_ls.remove("unital")
    ax_ls.append("left unit")
    ax_ls.append("right unit")
  if "left unit" in ax_ls or "right unit" in ax_ls: unit if unit else "1"
  base = ", ".join(ax_ls)
  ax = [e.replace("*",op) for e in axioms(base)]
  if unit: ax = [e.replace("1",unit) for e in ax]
  return ax

def iuband(op = "*", inv = "'", unit = "1"):
  ax = monoid("idempotent",op,unit) + axioms('involutive')
  return ax

def lattice(props = False, meet = "^",join = 'v', bounded = False, bot = False, top = False):
  if props:
    s = split_words(props)
    if "bounded" in s:
      s.remove("bounded")
      bounded = True
    if "distributive" in s:
      s.remove("distributive")
      s.append("left distributive")
    if bounded:
      bot = bot if bot else "0"
      top = top if top else "1"
  msl = band(meet, "commutative"+ ", unital"*(top != False),top)
  jsl = band(join, "commutative"+ ", unital"*(bot != False),bot)
  a = axiom_dict["left absorbing"]
  abs = [a.replace("*",meet).replace("+",join), a.replace("*",join).replace("+",meet)]
  ax = msl + jsl + abs
  if props:
    props = ", ".join(s)
    ax += axioms(props)
  return [e.replace("*",meet).replace("+",join) for e in ax]



def Kleene(op = "^", inv = "'", unit = "1"):
  ax = monoid("idempotent, commutative",op,unit) + axioms()

In [ ]:
def add_opst(models,eqs,cnt_model = []):
  def term_def_ops(model_cl,eqs,cnt=[]):
    tout=[]
    if model_cl == []:
      return []
    else:
      for model in model_cl:
        a = p9(model.diagram("") + eqs,cnt,1000,0,model.cardinality)
        if a != [] and type(a[0]) != str:
          tout += a
      return tout
      # return [prover9(model.diagram("") + eqs,[],1000,0,model.cardinality,one = True)[0] for model in model_cl]
  if type(models) == Model:
    models = [models]
    #return term_def_ops(models,eqs)
  if type(models) == str:
    models = []
  if type(models) == list and (models == [] or type(models[0]) == Model):
    return term_def_ops(models,eqs, cnt_model)
  elif type(models) == list and type(models[0]) == list and (models[0] == [] or type(models[0][0]) == Model):
    out = [[],[1]]
    for i in range(2,len(models)):
      print(i)
      out += [add_opst(models[i],eqs,cnt_model)]
    return out

# LaTeX tables display

In [ ]:
import string

def letters(n):
    alphabet = string.ascii_lowercase
    result = []

    # Loop to handle cases when n exceeds the alphabet length
    for i in range(n):
        letter = alphabet[i % 26]
        subscript = i // 26  # Determines if a subscript is needed
        if subscript > 0:
            result.append(f"{letter}_{{{subscript}}}")
        else:
            result.append(letter)

    return result

def neg_names(n):
    alphabet = string.ascii_lowercase
    result = []

    base_n = n if n % 2 == 0 else n - 1

    # Loop to handle cases when base_n exceeds the alphabet length
    for i in range(base_n // 2):
        letter = alphabet[i % 26]
        subscript = i // 26  # Determines if a subscript is needed

        if subscript > 0:
            result.append(f"{letter}_{{{subscript}}}'")
            result.append(f"{letter}_{{{subscript}}}")
        else:
            result.append(f"{letter}' ")
            result.append(letter)


    # If n is odd, add the next unused letter (without a negation)
    if n % 2 != 0:
        next_letter = alphabet[base_n // 2 % 26]
        subscript = base_n // 2 // 26
        if subscript > 0:
            result.append(f"{next_letter}_{{{subscript}}}")
        else:
            result.append(next_letter)

    return result

import sympy as sp
from IPython.display import display, Math


def unary_tab(input_list):
    return [[elem] for elem in input_list]

def tex_table(arr,op = "*",elements = [],tex = False,disp = True):
    if elements == []:
      elements = range(0,len(arr))
    if op == "v":
      op = "\lor "
    if op == "^":
      op = "\land "
    if op == "\ ":
      op = r"\backslash "
    if op == "\\":
      op = r"\backslash "
    if op == "/ ":
      op = "\slash "
    if op == "//":
      op = "\slash "
    if op == "*":
      op = "\cdot "
    if type(arr[0]) == int:
      dim = 0
      arr = unary_tab(arr)
    elif type(arr[0]) == list:
      dim = len(arr[0])
    # Create the LaTeX table header
    latex_str = r"\begin{array}{c|" + "c" * dim + "}\n"

    # Add the first row for the column headers (operands)
    # header_row = op + " & " + " & ".join(map(str, elements)) + r" \\\hline" + "\n"
    header_row = op + " & " + " & ".join(elements) + r" \\\hline" + "\n"
    latex_str += header_row

    # Fill the table rows
    for i, row in enumerate(arr):
        Row = [elements[j] for j in row]
        # latex_row = str(elements[i]) + " & " + " & ".join(map(str, Row)) + r" \\" + "\n"
        latex_row = elements[i] + " & " + " & ".join(Row) + r" \\" + "\n"
        latex_str += latex_row

    # End the LaTeX array
    latex_str += r"\end{array}"

    # Display the LaTeX table
    if disp:
      display(Math(latex_str))
    if tex == True:
      print(f"LaTeX code:\n{latex_str}")
      print(" ")
    return latex_str

def names_with_constants(N,constants,neg_tick = False):
  K = N - len(constants)
  if neg_tick == True:
    Klist = neg_names(K)
  else:
    Klist = letters(K)
  elements = constants + Klist
  return elements



def op_tables_tex(model,ops = [], elements_list = [], tex = False):
  N = model.cardinality
  Tex = []
  # if type(elements_list) == dict:
  #   elements = [elements_list[i] for i in range(N)]
  # else:
  elements = elements_list if not(elements_list == []) else [str(i) for i in range(N)]
  if ops == []:
    ops = list(model.operations.keys())
  for op in ops:
    array = model.operations[op]
    latex_str = tex_table(array,op,elements,tex)
    Tex += [latex_str]

def text2tex(str):
  print(f"LaTeX code:\n{str}")

<>:59: SyntaxWarning: invalid escape sequence '\l'
<>:61: SyntaxWarning: invalid escape sequence '\l'
<>:62: SyntaxWarning: invalid escape sequence '\ '
<>:67: SyntaxWarning: invalid escape sequence '\s'
<>:69: SyntaxWarning: invalid escape sequence '\s'
<>:71: SyntaxWarning: invalid escape sequence '\c'
<>:59: SyntaxWarning: invalid escape sequence '\l'
<>:61: SyntaxWarning: invalid escape sequence '\l'
<>:62: SyntaxWarning: invalid escape sequence '\ '
<>:67: SyntaxWarning: invalid escape sequence '\s'
<>:69: SyntaxWarning: invalid escape sequence '\s'
<>:71: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_258/4203245047.py:59: SyntaxWarning: invalid escape sequence '\l'
  op = "\lor "
/tmp/ipykernel_258/4203245047.py:61: SyntaxWarning: invalid escape sequence '\l'
  op = "\land "
/tmp/ipykernel_258/4203245047.py:62: SyntaxWarning: invalid escape sequence '\ '
  if op == "\ ":
/tmp/ipykernel_258/4203245047.py:67: SyntaxWarning: invalid escape sequence '\s'
  op = "\slash "

# saving models

In [ ]:
import json
def to_dict(self):
    """
    Return a JSON‐serializable dict representing this instance.
    Here, all attributes (int, dict-of-lists, etc.) are already JSON‐friendly,
    so we can just package them up in a new dict.
    """
    return {
        "cardinality": self.cardinality,
        "index": self.index,
        "operations": self.operations
    }

def from_dict(d):
    """
    Given a dict (as returned by to_dict), reconstruct a Model.
    """
    return Model(
        cardinality=d["cardinality"],
        index=d["index"],
        operations=d["operations"]
    )




def save_models(models,file_name = "", download = False):
  models = models[2:]
  if file_name == "":
    from datetime import datetime
    now = datetime.now() # current date and time
    file_name = "models_"+now.strftime("%Y/%m/%d/%H:%M")
  filepath = "/content/" + file_name +".json"
  data = [
      [ to_dict(m) for m in sublist ]
      for sublist in models
  ]
  with open(filepath, "w") as f:
      # no indent ⇒ arrays like [0,1,2,…] stay on one line
      json.dump(data, f)
  if download:
    from google.colab import files
    files.download(f"{file_name}.json")


def load_models(file_name):
    if file_name[-5:] == ".json":
      file_name = file_name[:-5]
    filepath = "/content/" + file_name +".json"
    with open(filepath, "r") as f:
        raw = json.load(f)
    # raw is a list of lists of dicts; rebuild Models
    return [[],[1]] + [
        [ from_dict(d) for d in sublist ]
        for sublist in raw
    ]


# save_models(Mc14, "McCarthy14")
# Mc14 = load_models("McCarthy14")
# show_t(C,"+",nh)

In [ ]:
# !pip install dot2tex
# import dot2tex
# import logging

def fast_graph_to_tikz(graph_or_dot,
                       prog: str    = 'dot',
                       texmode: str = 'math',
                       fmt: str     = 'tikz',
                       figonly: bool= True,
                       autosize: bool=True,
                       crop: bool    = True,
                       graphstyle: str = None) -> str:
    """
    Convert a Graphviz graph (or raw DOT string) to TikZ code—all in memory.

    Returns the TikZ picture code as a string.
    """
    logging.getLogger('dot2tex').setLevel(logging.ERROR)
    # 1) pull out the DOT source
    if hasattr(graph_or_dot, 'source'):
        dot_src = graph_or_dot.source
    elif isinstance(graph_or_dot, str):
        dot_src = graph_or_dot
    else:
        raise TypeError("Expected a graphviz.Graph/Digraph or a DOT string")
    # with open(os.devnull, 'w') as devnull: #This is to removing printing from Con
    #   with contextlib.redirect_stdout(devnull):
    # # 2) call dot2tex.dot2tex in‐memory
    #     st = dot2tex.dot2tex(
    #                           dot_src,
    #                           format     = fmt,
    #                           prog       = prog,
    #                           texmode    = texmode,
    #                           figonly    = figonly,
    #                           autosize   = autosize,
    #                           crop       = crop,
    #                           graphstyle = graphstyle)#[1:]
    # return st[1:]
    return dot2tex.dot2tex(
        dot_src,
        format     = fmt,
        prog       = prog,
        texmode    = texmode,
        figonly    = figonly,
        autosize   = autosize,
        crop       = crop,
        graphstyle = graphstyle  # e.g. "[scale=1.5,transform shape]"
    )[1:]


def m4diag_tex(models, ops, names, tables = [], cong = False,
             prog: str    = 'dot',
             texmode: str = 'math',
             fmt: str     = 'tikz',
             figonly: bool= True,
             autosize: bool=True,
             crop: bool    = True,
             graphstyle: str = "scale = .5, font = \small, every node/.style={minimum size=.5cm}, inner sep = 1pt") -> str:
  # out = []

  i = -1
  # st = ""
  st = r"\begin{enumerate}" + " \n"
  for x in models:
    if names == "": elements =[]
    else:
      elements = names(x)
      # elements_f = globals().get(name_func)
      # if not callable(elements_f):
      #   raise NameError(f"No callable named {name_func!r} in globals()")
      #   # elements = []
      # else:
      #   elements = elements_f(x)
    i+=1
    # st+=str(i)
    st += f"\item[({str(i)})]"
    unaryRel=""
    uR = x.relations[unaryRel] if unaryRel!="" else [0]*x.cardinality
    for op in tables:
      # t = tex_table(a.operations[op],op = op,elements = names(a),disp=False)
      # st += "$"+tex_table(x.operations[op],op = op,elements = names(x),disp=False)+r"$ \quad" +"\n"
      st += "$"+tex_table_n(x.operations[op],op = op,elements = names(x),blocks=fib_sets(x),disp=False)+r"$ \quad" +"\n"
    for s in ops:
            t = s[:-1] if s[-1] == 'd' else s #(s[-1]=='d' or s[-1]=='r')else s
            if t in x.operations.keys():
                G=hasse_diagram_t(x.operations[t],t,False,s[-1]=='d',uR,elements)
            elif t in x.relations.keys():
                G=hasse_diagram_t(x.relations[t],t, True, s[-1]=='d',uR,elements)
            st += fast_graph_to_tikz(G,
                       prog=prog,
                       texmode=texmode,
                       fmt=fmt,
                       figonly=figonly,
                       autosize=autosize,
                       crop=crop,
                       graphstyle=graphstyle)
            st += r"\quad" + "\n"
    # st +=  kl_po_props(x)+" \n"
    if cong:
      cL = poset2model(Con_lat(x))
      CL=hasse_diagram_t(cL.relations["<="],t, True, s[-1]=='d',uR,[])
      st += fast_graph_to_tikz(CL,
                       prog=prog,
                       texmode=texmode,
                       fmt=fmt,
                       figonly=figonly,
                       autosize=autosize,
                       crop=crop,
                       graphstyle=graphstyle)
      st += r"\quad" + "\n"
      st += con_tex(x,False)
  st += "\end{enumerate} \n"
  return st




def show_tex(K,ops=[],names = "",tables = [],cong = False): # show a list of Mace4 models using graphviz or show a set of subsets or partitions
  if type(K)==Model: K=[K]
  if type(K)==str: K=[]
  if type(K)==list and len(K)>0 and type(K[0])==Model:
    if type(ops) == str:
      st = ops
    else:
      if ops==[]:
        ops = bin_ops(K[0])
      # ops=[x.strip() for x in ops]
      # st=" ".join(ops)
    st = m4diag_tex(K,ops,names, tables, cong)
    return st
  elif type(K) == list and len(K)> 2 and type(K[2])==list: # and type(K[2][0]) == Model:
    preamble = (
        r"\documentclass{article}" + "\n"
        r"\usepackage[margin=.5in]{geometry}" + "\n"
        r"\usepackage{hyperref}" +"\n"
        r"\usepackage{amsmath}" +"\n"
        r"\usepackage{arydshln}" +"\n"
        # r"\documentclass{standalone}" + " \n"
        "\\usepackage{tikz}\n"
        r"\usepackage{titlesec}" + "\n"
        r"\titleformat{\subsection}{\normalfont\large\bfseries}{}{0pt}{}" + "\n"
        r"\titleformat{\subsection}{\normalfont\large\bfseries}{}{0pt}{}" +"\n"
        r"\arraycolsep=2pt" + "\n"
        r"\def\arraystretch{1}" + "\n"
        r"\begin{document}" + "\n"
        r"\tableofcontents" +"\n"
        r"\tikzset{baseline=(current bounding box.center)}" + "\n"
    )
    postamble = "\n\\end{document}\n"
    out = "\subsection{Info} \n"
    c = 2
    for i in range(2,len(K)):
      m_lab = ": 1 model} \n" if len(K[i]) == 1 else ": "+str(len(K[i]))+" models} \n"
      # out += "\subsection{Cardinality "+str(K[i][0].cardinality)+ m_lab
      if type(K[i][0]) == Model:
        c = K[i][0].cardinality
      # out += "\subsection{Cardinality "+str(i)+ m_lab
      out += "\subsection{Cardinality "+str(c)+ m_lab
      c += 1
      # print("-"*20 + f"Cardinality {K[i][0].cardinality}"+ "-"*20)
      if len(K[i]) > 0:
        out += show_tex(K[i],ops,names,tables,cong)
    return preamble + out + postamble
  elif type(K)==frozenset: m4diag_t([poset2model(K)],name_func = names)
  elif type(K)==list: m4diag_t([poset2model(x) for x in K],name_func = names)

def con_tex(alg, show = True):
  cons = Con_lat(alg)
  eqvc = [sorted([list(y) for y in x]) for x in cons]
  tex = r"$\begin{array}{r l}" + " \n"
  for i,th in enumerate(eqvc):
    tex += str(i) + ": & "
    for ec in th:
      tex += r"\{"
      for e in ec:
        tex += f"{nt(alg)[e]}, "
      tex = tex[:-2] + r"\}, "
    tex = tex[:-2] + r"\\" + "\n"
  tex += r"\end{array}$"
  if show:
    show_t(alg,["+","v"],nh)
    show(cons,"<=")
    display(Math(tex))
  else: return tex


# print(show_tex(Msw,["v"],nt,["*"],True))

# congruence classes


In [ ]:
def alg_quotient(alg,cng,names):
  ax = alg.positive_diagram('a')
  new_names = {}
  for i,ec in enumerate(cng):
    ax += [f"a{ec[0]} = {i}"]
    new_names.update({i : names(alg)[ec[0]]})
    # st = ",".join([str(j) for j in ec])
    # new_names.update({i : [st,""]})
    if len(ec) > 1:
      for i in ec[1:]:
        ax += [f"a{ec[0]} = a{i}"]
  alg_mod_cng = prover9(ax,[],10, cardinality=len(cng),one = True)[0]
  return [alg_mod_cng, new_names]


def cng_dict(cL):
  # scL = sorted(cL, key = len)
  rng = range(cL.cardinality)
  # names = {scL[0] : ["Delta",""], scL[-1] : ["nabla",""]}
  names = {0 : ["0",""], rng[-1] : ["T",""]}
  # for i,ec in enumerate(scL[1:-1]):
  for i in rng[1:-1]:
    names.update({i : [str(i),""]})
  return names



def Con_t(alg, names = None,op = ["*"] , tabels = [] , quotient = False, tex_out = False):

  cons = Con_lat(alg)
  eqvc = [sorted([list(y) for y in x]) for x in sorted(cons,key = len)]
  if names: elms = name_format(names(alg),"tex")
  elif type(names) == None: elms = range(A.cardinality)
  tex = r"$\begin{array}{r l}" + " \n"
  for i,th in enumerate(eqvc):
    tex += str(i) + ": & "
    # for ec in th:
    #   tex += r"\{"
    #   for e in ec:
    #     # tex += f"{nt(alg)[e]}, "
    #     tex += f"{names[e]}, "
    #   tex = tex[:-2] + r"\}, "
    for ec in th:
      # tex += r"\{"
      for e in ec:
        # tex += f"{nt(alg)[e]}, "
        tex += f"{elms[e]}, "
      tex = tex[:-2] + r"\mid "
    tex = tex[:-5] + r"\\" + "\n"
  tex += r"\end{array}$"

  if not tex_out:
    show_t(alg,op,names,tabels)
    show_t(cons)
    display(Math(tex))
  else: return tex
  if quotient:
    Qls = []
    for c in eqvc[1:]:
      # print(c)
      Qls.append(alg_quotient(alg,c,names))
    Qs = [Model(1,0,relations = { "<=" : [[0]]})] + [Model(Qls[i][0].cardinality, i, Qls[i][0].operations, Qls[i][0].relations) for i in range(len(eqvc[1:]))]
    def new_names(a):
      return Qls[a.index][1]
    show_t(Qs,["<="],new_names)
    # show_t(Qs,["<="],names)
    # return Qs






In [ ]:
def Con_names(A,ops =[],names=[]):
  N = range(len(ops))
  elements = names
  if elements == []:
      elements = N
  if type(A)==Model: return frozenset(eqrel2partition(elements[x]) for x in compatiblepreorders(A,False,True))
  return [frozenset(eqrel2partition(elements[x]) for x in compatiblepreorders(y,False,True)) for y in A]

def showConClass_names(md,ops=[],names=[],op_table = 0, name = []):
  print("---------------------------Model ["+str(md.cardinality)+"]"+str(name)+"-----------------------------------------------")
  print(show_names(md,ops,names))
  if op_table == 1:
    print(md)
  print("-----------------------------Congruence Lattice----------------------------------")
  cons = Con_names(md,ops,names)
  print(show_names(cons,ops,names))
  eqvc = [sorted([list(y) for y in x]) for x in cons]
  for i in range(0,len(eqvc)):
    print(i,":",eqvc[i])
  print("------------------------------End------------------------------------------------")
  print("---------------------------------------------------------------------------------")


# UAcalc out


In [ ]:
import os
import shutil


def uacalc_zip(models, nametag = "A" , folder_name = "UAlgebras", download = True , files_contents = dict()):
  if type(models)==Model: models=[models]
  if type(models[0]) == Model:
    Tot = len(models)
    Tot_mag = len(str(len(models)))
    for i in range(Tot):
      pad = "0"*(len(str(Tot)) - len(str(i)))
      card = models[i].cardinality
      name = f"{nametag}{card}_{pad}{i}"
      text = uacalc_format(models[i],name)
      filename = f"{name}.ua"
      # print(filename)
      files_contents.update({filename : text})
      return files_contents
    return files_contents
  elif type(models) == list and len(models)> 2 and type(models[2])==list: # and type(K[2][0]) == Model:
    for i in range(len(models)):
      if len(models[i])>0 and type(models[i][0]) == Model:
        files_contents = uacalc_zip(models[i], nametag, folder_name, files_contents)
  os.makedirs(folder_name, exist_ok=True)
  for fname, content in files_contents.items():
      path = os.path.join(folder_name, fname)
      with open(path, "w") as f:
          f.write(content)
  shutil.make_archive(folder_name, 'zip', folder_name)
  if download:
    from google.colab import files
    files.download(f"{folder_name}.zip")

In [ ]:
import os
import shutil


def uacalc_zip(models, nametag = "A" , folder_name = "UAlgebras", download = True,flag = True):
  os.makedirs(folder_name, exist_ok=True)
  if type(models)==Model: models=[models]
  if type(models[0]) == Model:
    Tot = len(models)
    Tot_mag = len(str(len(models)))
    for i in range(Tot):
      pad = "0"*(len(str(Tot)) - len(str(i)))
      card = models[i].cardinality
      name = f"{nametag}{card}_{pad}{i}"
      content = uacalc_format(models[i],name)
      fname = f"{name}.ua"
      path = os.path.join(folder_name, fname)
      with open(path, "w") as f:
          f.write(content)
  elif type(models) == list and len(models)> 2 and type(models[2])==list: # and type(K[2][0]) == Model:
    for i in range(len(models)):
      if len(models[i])>0 and type(models[i][0]) == Model:
        uacalc_zip(models[i], nametag, folder_name, download,False)
    flag = True
  #
  if flag:
    shutil.make_archive(folder_name, 'zip', folder_name)
    if download:
      from google.colab import files
      files.download(f"{folder_name}.zip")

# defined congruence test

In [ ]:
def op_arity(ls,count = 0):
      if type(ls) != list: return count
      elif type(ls) == list:
        count = op_arity(ls[0],count + 1)
      return count

from ast import Compare
def is_con(ass, con_ass = ["C(x,y) <-> x = y"], Cg = "C", m4time = 4, p9time = 300,check_equiv = True, op_dict = {}):
  if type(con_ass) == str: con_ass = [con_ass]
  test = p9(ass,["x = y"],4,60)[0]
  if type(test) == Proof:
    print("Inconsistent assumptions")
    return test
  elif type(test) == Model:
    ops = test.operations.keys()

    op_dict = {op : op_arity(test.operations[op]) for op in ops if op_arity(test.operations[op]) > 0}
  elif op_dict == {}: return print("You must input op_dict, a dictionary arities of the ops.")
  equiv = [f"{Cg}(x,x)",
           f"{Cg}(x,y) -> {Cg}(y,x)",
           f"{Cg}(x,y) & {Cg}(y,z) -> {Cg}(x,z)"]
  out = []
  F = []
  T = []
  flag = None
  if check_equiv:
    flag = True
    for e in equiv:
      t = p9(ass + con_ass + T, [e], m4time, p9time)
      if t == []:
          print(f"-- {e} no proof after {p9time}s, no CM after {m4time}s")
          out.append([])
          flag = None
      elif type(t[0]) == Model:
        print(f"-- {e} is false")
        F += [e]
        print(t[0])
        # show_t(t[0],["v","+"],nh)
        out += t
        flag = False
      elif type(t[0]) == Proof:
        print(f"-- {e} holds")
        # T += [e]
        out += t
        # flag = flag and True
    if flag:
      print("-C is an equivalence relation.")

  flag2 = True
  for op in op_dict.keys():
    ar = op_dict[op]
    antec = [Cg + f"(x{j} , y{j})" for j in range(ar)]
    conseq = Cg + "(" + " ,".join([op + "(" + " ,".join([l + str(i) for i in range(ar)]) + ")" for l in ["x","y"]]) + ")"
    op_qe = " & ".join(antec) + " -> " + conseq
  #for e in compat:
    # t = p9(ass + con_ass + T, [e], m4time, p9time)
    t = p9(ass + con_ass + T, [op_qe], m4time, p9time)
    if t == []:
        print(f"-- {op} compatibility has no proof after {p9time}s, no CM after {m4time}s")
        flag2 = None
        out.append([])
    elif type(t[0]) == Model:
      print(f"-- {op} compatibility is FAILS")
      F += [op_qe]
      print(t[0])
      # show_t(t[0],["v","+"],nh)
      out += t
      flag2 = False
    elif type(t[0]) == Proof:
      print(f"-- {op} is compatible with {Cg}")
      # T += [e]
      out += t
  if flag2:
    print(f"-{Cg} preserves the operations.")
  if flag and flag2:
    print("{Cg} is a congruence.")
  return out

# Proof Simplification


In [ ]:
def p9latex(pf, latex=False):
    # print a proof in latex format (slightly more readable)
    from IPython.display import display, Math
    import re
    if type(pf) == Model or type(pf[0]) == Model:
      print(pf)
      return
    la = [str(li[0])+"\\quad "+re.sub(r"c(\d)",r"c_\1",li[1]).\
          replace(" ^ "," \\land ").replace(" v "," \\lor ").\
          replace(" * ","").replace("<=","\\le ").replace("!=","\\ne ").\
          replace("\\ ","\\backslash ").replace(" <->","\\iff").\
          replace(" ->","\\implies").replace("exists","\\exists").\
          replace("# label(non_clause)","").replace("# label(goal)","\\quad(goal)").\
          replace("&","\\ \\&\\ ").replace("|","\\quad\\mbox{or}\\quad").replace("$F","F")\
          +"\\quad "+str(li[2]) for li in pf[0].proof]
    if latex: return "$"+"$\n\n$".join(la)+"$"
    else:
        for st in la: display(Math(st))

In [ ]:
#De Morgan general
import re
def DeMorgan_simplify(expr, DM_dict = ["'","*","+"]):
    """Simplifies the expression according to the d_transformation rules."""
    def d_tokenize(expr):
        """d_tokenizes the input string into variables, constants, operators, and parentheses."""
        expr = expr.replace(' v ',' V ')
        expr = expr.replace(' ', '')  # Remove all spaces
        tokens = re.findall(r"[a-z0-9]+|[()V^+*']", expr)
        return tokens

    def d_parse(tokens, DM_dict = ["'","*","+"]):
        """d_parses tokens into an expression tree."""
        neg = DM_dict[0]
        mult = DM_dict[1]
        add = DM_dict[2]
        def d_parse_expression(index):
            """Handles expressions with '+' operators."""
            node, index = d_parse_term(index)
            while index < len(tokens) and not tokens[index] in ["(",")",neg,mult]:#tokens[index] == add:
                op = tokens[index]
                index += 1  # Move past '+'
                right_node, index = d_parse_term(index)
                node = (op, node, right_node)
            return node, index


        def d_parse_term(index):
            """Handles expressions with '*' operators."""
            node, index = d_parse_factor(index)
            while index < len(tokens) and tokens[index] == mult:
                op = tokens[index]
                index += 1  # Move past '*'
                right_node, index = d_parse_factor(index)
                node = (op, node, right_node)
            return node, index

        def d_parse_factor(index):
            """Handles variables, constants, negations, and parentheses."""
            if index >= len(tokens):
                raise ValueError("Unexpected end of tokens")
            token = tokens[index]
            if token == '(':
                index += 1  # Move past '('
                node, index = d_parse_expression(index)
                if index >= len(tokens) or tokens[index] != ')':
                    raise ValueError("Expected ')'")
                index += 1  # Move past ')'
            elif re.match(r'[a-z0-9]+', token):
                node = token
                index += 1  # Move past variable or constant
            else:
                raise ValueError(f"Unexpected token: {token}")

            # Handle postfix negation operator "'"
            while index < len(tokens) and tokens[index] == neg:
                node = (neg, node)
                index += 1  # Move past "'"
            return node, index

        node, index = d_parse_expression(0)
        if index != len(tokens):
            raise ValueError("Extra tokens after parsing")
        return node

    def d_transform(node,DM_dict = ["'","*","+"]):
        """Recursively applies d_transformation rules to the expression tree."""
        neg = DM_dict[0]
        mult = DM_dict[1]
        add = DM_dict[2]
        if isinstance(node, str):
            # Rule (1): Variable or constant remains as is
            return node
        elif node[0] == neg:
            # Negation operator
            subexpr = node[1]
            d_transformed_subexpr = d_transform(subexpr,DM_dict)
            if d_transformed_subexpr == '0':
                # Rule (2): 0' -> 1
                return '1'
            elif d_transformed_subexpr == '1':
                # Rule (3): 1' -> 0
                return '0'
            elif isinstance(d_transformed_subexpr, tuple) and d_transformed_subexpr[0] == neg:
                # Rule (4): Double negation
                return d_transform(d_transformed_subexpr[1], DM_dict)
            elif isinstance(d_transformed_subexpr, tuple):
                op = d_transformed_subexpr[0]
                if op == mult:
                    # Rule (5): (Exp1 * Exp2)' -> Exp1' + Exp2'
                    left = d_transform((neg, d_transformed_subexpr[1]), DM_dict)
                    right = d_transform((neg, d_transformed_subexpr[2]), DM_dict)
                    return (add, left, right)
                elif op == add:
                    # Rule (6): (Exp1 + Exp2)' -> Exp1' * Exp2'
                    left = d_transform((neg, d_transformed_subexpr[1]), DM_dict)
                    right = d_transform((neg, d_transformed_subexpr[2]), DM_dict)
                    return (mult, left, right)
                else:
                    # Negation of other expressions
                    return (neg, d_transformed_subexpr)
            else:
                # Negation of a variable or constant
                return (neg, d_transformed_subexpr)
        else:
            # Rule (1): Recursively d_transform sub-expressions
            op = node[0]
            left = d_transform(node[1], DM_dict)
            right = d_transform(node[2], DM_dict)
            return (op, left, right)

    def tree_to_string(node,DM_dict = ["'","*","+"], outermost=True):
        """Converts the expression tree back into a string with parentheses."""
        if isinstance(node, str):
            return node
        elif node[0] == DM_dict[0]:
            subexpr_str = tree_to_string(node[1], DM_dict,outermost=False)
            # Include parentheses around sub-expression if it's an operation
            if isinstance(node[1], tuple):
                return f"({subexpr_str})'"
            else:
                return f"{subexpr_str}'"
        else:
            op = node[0]
            left_str = tree_to_string(node[1], DM_dict,outermost=False)
            right_str = tree_to_string(node[2], DM_dict,outermost=False)
            expr_str = f"{left_str} {op} {right_str}"
            if not outermost:
                expr_str = f"({expr_str})"
            return expr_str

    if DM_dict[2] ==  "v":
      DM_dict = [DM_dict[0], DM_dict[1], "V"]
    tokens = d_tokenize(expr)
    tree = d_parse(tokens,DM_dict)
    d_transformed_tree = d_transform(tree,DM_dict)
    result = tree_to_string(d_transformed_tree, DM_dict)
    result = result.replace('V','v')
    return result

# Example usage
input_expr = "(x v y)' * (x ^ y)' "
simplified_expr = DeMorgan_simplify(input_expr,DM_dict = ["'","^","v"])
print("Simplified Expression:", simplified_expr)

Simplified Expression: (x' ^ y') * (x' v y')


In [ ]:
# Associative operations
import re

# Define operator precedence
precedence = {'+': 1, '*': 2, "'": 3}

def tokenize(expr):
    """Tokenizes the input string into variables, constants, operators, and parentheses."""
    expr = expr.replace(' ', '')  # Remove spaces
    tokens = re.findall(r"[a-zA-Z0-9]+|[()+*']^", expr)
    return tokens

def parse(tokens):
    """Parses tokens into an expression tree."""
    def parse_expression(index):
        node, index = parse_term(index)
        while index < len(tokens) and tokens[index] == '+':
            op = tokens[index]
            index += 1
            right_node, index = parse_term(index)
            node = (op, node, right_node)
        return node, index

    def parse_term(index):
        node, index = parse_factor(index)
        while index < len(tokens) and tokens[index] == '*':
            op = tokens[index]
            index += 1
            right_node, index = parse_factor(index)
            node = (op, node, right_node)
        return node, index


    def parse_factor(index):
        if index >= len(tokens):
            raise ValueError("Unexpected end of tokens")
        token = tokens[index]
        if token == '(':
            index += 1
            node, index = parse_expression(index)
            if index >= len(tokens) or tokens[index] != ')':
                raise ValueError("Expected ')'")
            index += 1  # Move past ')'
        elif re.match(r'[a-zA-Z0-9]+', token):
            node = token
            index += 1
        else:
            raise ValueError(f"Unexpected token: {token}")
        # Handle postfix negation operator "'"
        while index < len(tokens) and tokens[index] == "'":
            node = ("'", node)
            index += 1
        return node, index

    node, index = parse_expression(0)
    if index != len(tokens):
        raise ValueError("Extra tokens after parsing")
    return node

def associativity_rewrite(node, op):
    """Rewrites the expression tree by associating the given operator."""
    if isinstance(node, str):
        return node
    elif node[0] == "'":
        operand = associativity_rewrite(node[1], op)
        return ("'", operand)
    elif node[0] == op:
        # Flatten operands connected via op
        operands = flatten_op(node, op)
        # Apply associativity_rewrite to each operand
        new_operands = [associativity_rewrite(operand, op) for operand in operands]
        # Reconstruct the expression tree with flattened operands
        if len(new_operands) == 1:
            return new_operands[0]
        else:
            # Build the tree left-associatively
            result = new_operands[0]
            for operand in new_operands[1:]:
                result = (op, result, operand)
            return result
    else:
        # Operator is not the target op, apply recursively
        left = associativity_rewrite(node[1], op)
        right = associativity_rewrite(node[2], op)
        return (node[0], left, right)

def flatten_op(node, op):
    """Flattens the operands connected via the associative operator."""
    if isinstance(node, str) or node[0] != op:
        return [node]
    else:
        left_operands = flatten_op(node[1], op)
        right_operands = flatten_op(node[2], op)
        return left_operands + right_operands

def tree_to_string(node, parent_op=None):
    """Converts the expression tree back into a string with appropriate parentheses."""
    if isinstance(node, str):
        return node
    elif node[0] == "'":
        operand_str = tree_to_string(node[1], "'")
        # Include parentheses if needed based on precedence
        if isinstance(node[1], tuple) and precedence.get(node[1][0], 0) < precedence["'"]:
            operand_str = f"({operand_str})"
        return f"{operand_str}'"
    else:
        op = node[0]
        left_str = tree_to_string(node[1], op)
        right_str = tree_to_string(node[2], op)
        expr_str = f"{left_str} {op} {right_str}"
        # Include parentheses if the current operator has lower precedence than the parent operator
        if parent_op is not None and precedence[op] < precedence[parent_op]:
            expr_str = f"({expr_str})"
        return expr_str

def rewrite_associativity(expr, op):
    """Main function to rewrite the expression based on associativity of the given operator."""
    tokens = tokenize(expr)
    tree = parse(tokens)
    new_tree = associativity_rewrite(tree, op)
    result = tree_to_string(new_tree)
    return result

# Example usage
input_expr = "((x*((y+z) * u'))*w') + 1"
op = "*"
rewritten_expr = rewrite_associativity(input_expr, op)
print("Rewritten Expression:", rewritten_expr)

ValueError: Extra tokens after parsing

In [ ]:
import sympy as sp
from IPython.display import display, Math

import re

def TeX_expression(exp, op_index,pre="",post=""):
    # Build a dictionary from op_index for easy lookup
    op_dict = dict(op_index)

    # Regular expression patterns
    variable_pattern = r'[a-zA-Z]+[0-9]*'  # Matches letters followed by optional numbers
    operator_pattern = '|'.join(map(re.escape, op_dict.keys()))
    parenthesis_pattern = r'[()]'

    # Combined pattern to tokenize the expression
    token_pattern = f'{variable_pattern}|{operator_pattern}|{parenthesis_pattern}|.'

    # Tokenize the expression
    tokens = re.findall(token_pattern, exp)

    # Process each token
    processed_tokens = []
    for token in tokens:
        if re.match(variable_pattern, token):
            # Check if the variable has a number (subscript)
            match = re.match(r'^([a-zA-Z]+)([0-9]+)$', token)
            if match:
                # Variable with subscript
                var = match.group(1)
                subscript = match.group(2)
                processed_token = f'{var}_{{{subscript}}}'
            else:
                # Variable without subscript
                processed_token = token
            processed_tokens.append(processed_token)
        elif token in op_dict:
            # Replace operator with its LaTeX representation
            processed_tokens.append(op_dict[token])
        elif token in '()':
            # Keep parentheses as is
            processed_tokens.append(token)
        else:
            # Handle any other characters (e.g., spaces)
            processed_tokens.append(token)

    # Join the processed tokens
    latex_expr = ''.join(processed_tokens)
    return display(Math(pre + latex_expr + post))

# # Example usage
# exp = "((x*((y+z) * u12'))*w') + 1"
# op_index = [('*', r'\cdot '), ('+', '+')]  # Note: Added a space after \cdot for readability

# latex_output = TeX_expression(exp, op_index)
# print("LaTeX Expression:", latex_output)

def equation_simplify(eq,ops=["*"]):
  if "!=" in eq:
    E = eq.split("!=")
    rel = " != "
  elif "=" in eq:
    E = eq.split("=")
    rel = " = "
  out = []
  for exp in E:
    #exp = simplify_expression(exp)
    for op in ops:
      exp = rewrite_associativity(exp, op)
    out.append(exp)
  return out[0] + rel + out[1]

def Tex_eq_simp(eq,ops = ["*"],tex_index = [('*',r'\cdot')],pre="",post=""):
  tex_index.append((" != ", r"\neq "))
  exp = equation_simplify(eq,ops)
  return TeX_expression(exp,tex_index,pre,post)

eq = "(x+(y* z))' != ((x+y)* z)'+ 1'"
print(eq)
tex_index = [('*',''),('+','+')]
ops = ["*","+"]

Tex_eq_simp(eq,ops,tex_index)



In [ ]:
def remove_redundancies(P):
    # Step 1: Identify redundant entries (consecutive entries with the same string)
    i = 0
    while i < len(P) - 1:
        # Check for consecutive entries with the same string
        current_str = P[i][1]
        k = 0
        while i + k + 1 < len(P) and P[i + k + 1][1] == current_str:
            k += 1

        if k > 0:
            # Step 2: Remove redundant entries P[i+1], ..., P[i+k]
            del P[i+1:i+1+k]

            # Step 3: Relabel indices for entries after P[i]
            for j in range(i + 1, len(P)):
                P[j][0] = j + 1  # Update index to reflect new position
                updated_references = []
                for ref in P[j][2]:
                  if i < ref <= i + k:
                      updated_references.append(i)  # Redirect to the retained index
                  elif ref > i + k:
                      updated_references.append(ref - k)  # Adjust for removed indices
                  else:
                      updated_references.append(ref)  # Keep unchanged if before i
                P[j][2] = updated_references

            # # Step 4: Update reference lists
            # for entry in P:
            #     updated_references = []
            #     for ref in entry[2]:
            #         if i < ref <= i + k:
            #             updated_references.append(i)  # Redirect to the retained index
            #         elif ref > i + k:
            #             updated_references.append(ref - k - 1)  # Adjust for removed indices
            #         else:
            #             updated_references.append(ref)  # Keep unchanged if before i
            #     entry[2] = updated_references

        # Increment index to process the next set of entries
        i += 1

    return P

def TeX_proof(P,
              is_associative = [],
              is_DeMorgan = [],
              # is_associative = None,
              # is_DeMorgan = None,
              op_dict = [("*",""), ("+","+"), ("^",r"\land"),(" v ",r"\lor"),("/",r"\backslash")],):
  if isinstance(P, list):
    for i in range(len(P)):
      Pr = []
      pr = TeX_proof(P[i],is_associative, is_DeMorgan, op_dict)
      Pr.append(pr)
    return Pr
  if isinstance(P,Model):
    print("There is a countermodel")
    #print(P)
    #tex = op_tables_tex(P,list(P.operations.key()))
    #print(tex)
    return op_tables_tex(P,list(P.operations),range(P.cardinality))

  if isinstance(P,Proof):
    op_dict.append((" != ", r"\neq "))
    p=P.proof
    Pf = []
    Pre = []
    Post = []
    for l in range(len(p)):
      exp = p[l][1].split('#')[0]
      if "!=" in exp:
        Eq = exp.split("!=")
        rel = " != "
      elif "=" in exp:
        Eq = exp.split("=")
        rel = " = "
      out = []
      for e in Eq:
        if is_DeMorgan and not p[l][2] == []:
          e = DeMorgan_simplify(e, is_DeMorgan)
        for op in is_associative:
          e = rewrite_associativity(e,op)
          #e = equation_simplify(e,is_associative)
        out.append(e)
      line = out[0] + rel + out[1]
      #TeX_expression(line,op_dict,pre,post)
      Pf.append([p[l][0],line,p[l][2]])
    Pf = remove_redundancies(Pf)
    for ln in range(len(Pf) - 1):
      pre = str(Pf[ln][0]) + ".~"
      #exp = p[l][1].split('#')[0]
      exp_end = r"~\text{"
      I = range(len(Pf[ln][1].split('#')))
      for i in I[1:]:
        exp_end = exp_end + Pf[ln][1].split('#')[i]
      exp_end = exp_end + r"}"
      exp_end = exp_end.replace("_"," ")
      post = exp_end + ' \quad ' + str(Pf[ln][2])
      TeX_expression(Pf[ln][1],op_dict,pre,post)
    display(Math(str(Pf[len(Pf)-1][0]) + r".~\text{contradiction}~" + str(Pf[len(Pf)-1][2])))

    return Pf





In [ ]:
def exp_trans(expression, is_associative, is_DeMorgan,P):
  if "!=" in expression:
    Eq = expression.split("!=")
    rel = " != "
  elif "=" in expression:
    Eq = expression.split("=")
    rel = " = "
  else:
    return expression
  out = []
  for e in Eq:
    if is_DeMorgan and not P == []:
      e = DeMorgan_simplify(e, is_DeMorgan)
    for op in is_associative:
      e = rewrite_associativity(e,op)
      #e = equation_simplify(e,is_associative)
    out.append(e)
  return out[0] + rel + out[1]

#redun for quasieqs
def TeX_proof(P,
              is_associative = None,
              is_DeMorgan = None,
              op_dict = [("*",""), ("+","+"), ("^",r"\land"),(" v ",r"\lor"),("/",r"\backslash")],):
  if isinstance(P, list):
    for i in range(len(P)):
      Pr = []
      pr = TeX_proof(P[i],is_associative, is_DeMorgan, op_dict)
      Pr.append(pr)
    return Pr
  if isinstance(P,Model):
    print("There is a countermodel")
    #print(P)
    #tex = op_tables_tex(P,list(P.operations.key()))
    #print(tex)
    return op_tables_tex(P,list(P.operations),range(P.cardinality))

  if isinstance(P,Proof):
    op_dict.append((" != ", r"\neq "))
    p=P.proof
    Pf = []
    Pre = []
    Post = []
    logic_con = {"->":r" \implies ",
                 "<->":r" \iff ",
                 "|":r" ~\mathsf{or}~ ",
                 "&":r" ~\mathsf{and}~ "
                 }
    for l in range(len(p)):
      exp_p = p[l][1].split('#')
      exp = exp_p[0]
      is_goal = r"\text{goal}" if any("goal" in exp_p[i] for i in range(1,len(exp_p))) else ""
      if any(x in exp for x in logic_con.keys()):
        for op in [x for x in logic_con.keys() if x in exp]:
          Qexp = exp.split(op)
          lines = []
          for ex in Qexp:
            lines += [exp_trans(ex,is_associative,is_DeMorgan,p[l])]
            # line.append(exp_trans(ex,is_associative,is_DeMorgan))
          line = logic_con[op].join(lines)
      else:
        line = exp_trans(exp, is_associative,is_DeMorgan,p[l])
      #TeX_expression(line,op_dict,pre,post)
      Pf.append([p[l][0],line,p[l][2]])
    Pf = remove_redundancies(Pf)
    for ln in range(len(Pf) - 1):
      pre = str(Pf[ln][0]) + ".~"
      #exp = p[l][1].split('#')[0]
      exp_end = r"~\text{"
      I = range(len(Pf[ln][1].split('#')))
      for i in I[1:]:
        exp_end = exp_end + Pf[ln][1].split('#')[i]
      exp_end = exp_end + r"}"
      exp_end = exp_end.replace("_"," ")
      post = exp_end + ' \quad ' + str(Pf[ln][2])
      TeX_expression(Pf[ln][1],op_dict,pre,post)
    display(Math(str(Pf[len(Pf)-1][0]) + r".~\text{contradiction}~" + str(Pf[len(Pf)-1][2])))

    return Pf


In [ ]:
ax = McCarthy()+["x + y = (x'*y')'"]
goal = ['(x*y)*x = x*y']
Pr = prover9(ax, goal , prover_seconds=600)
P = TeX_proof(Pr,["*","+"],["'","*","+"])

# McCarthy


In [ ]:
def SLat(model):
  p = model.operations["*"]
  n = model.operations["'"]
  elem = range(0,model.cardinality)
  return [p[i][n[i]] for i in elem if p[i][n[i]]==i]

def Blocks(model):
  p = model.operations["*"]
  n = model.operations["'"]
  elem = range(0,model.cardinality)
  SL = SLat(model)
  blks = []
  for i in SL:
    blks += [[ a for a in elem if p[a][n[a]] == i]]
  Blocks = [[[i,n[i]]] for i in SL ]
  for i in range(len(SL)):
    for a in blks[i]:
      if not([a,n[a]] in Blocks[i] or [n[a],a] in Blocks[i]):
        Blocks[i] += [[a,n[a]]]
  return Blocks


import string
def block_letters(n):
    alphabet = string.ascii_lowercase
    result = []
    # Loop to handle cases when n exceeds the alphabet length
    for i in range(n):
        letter = alphabet[i % 26]
        subscript = i // 26  # Determines if a subscript is needed
        if subscript > 0:
            #result.append(f"{letter}_{{{subscript}}}")
            result.append(letter * (subscript+1))
        else:
            result.append(letter)
    return result

def add_names(block_pair,let_pair,n_dict = {}):
  n_dict.update({block_pair[0]:let_pair[0]})
  n_dict.update({block_pair[1]:let_pair[1]})
  return n_dict



def block_dict(model):
  blocks = Blocks(model)
  n = len(blocks)
  N = range(n)
  SL = SLat(model)
  # letters = block_letters(n)

  if len(blocks[0]) == 1:
    lets = ["none"] + block_letters(n-1)
  else:
    lets = block_letters(n)
  shift = 0
  letters = []
  for i in N:
    B = blocks[i][0]
    lab = lets[i + shift]
    if B[0] == B[1]:
      shift = -1
      lab = "FP"
    letters += [lab]
  names = {0 : '0', 1 : '1'}
  for i in N[1:]:
    #names = add_names(blocks[i][0], ['0_'+letters[i],'1_'+letters[i]], names)
    # names = add_names(blocks[i][0], [letters[i]+'0',letters[i]+'+1'], names)
    names = add_names(blocks[i][0], ["<0<SUB>"+letters[i]+"</SUB>>",'<1<SUB>'+letters[i]+"</SUB>>"], names)
    if blocks[i][0][0] == blocks[i][0][1]:
      # names.update({blocks[i][0][0]: "<&epsilon;<SUB>"+letters[i]+"</SUB>>"})
      names.update({blocks[i][0][0]: "<&epsilon;>"})
  for i in N:
    B = blocks[i]
    insideB = range(1,len(B))
    if len(insideB) == 1:
      names = add_names(blocks[i][1], [letters[i],letters[i]+"'"], names)
    if len(insideB) > 1:
      for j in insideB:
        # names = add_names(blocks[i][j], [letters[i]+'_'+str(j),letters[i]+'_'+str(j)+"'"],names)
        names = add_names(blocks[i][j], [f'<{letters[i]}<SUB>{str(j)}</SUB>>',f"<{letters[i]}'<SUB>{str(j)}</SUB>>"],names)
  return names


# def fibers(model):
#   blocks = Blocks(model)
#   n = len(blocks)
#   N = range(n)
#   SL = SLat(model)
#   # letters = block_letters(n)
#   if len(blocks[0]) == 1:
#     lets = ["none"] + block_letters(n-1)
#   else:
#     lets = block_letters(n)
#   shift = 0
#   letters = []
#   for i in N:
#     B = blocks[i][0]
#     lab = lets[i + shift]
#     if B[0] == B[1]:
#       shift = -1
#       lab = "FP"
#     letters += [lab]
#   names = {0 : ['0',""], 1 : ['1',""]}
#   for i in N[1:]:
#     names = add_names(blocks[i][0], [["0",letters[i]],["1",letters[i]]], names)
#     if blocks[i][0][0] == blocks[i][0][1]:
#       names.update({blocks[i][0][0]: ["epsilon",""]})#"<&epsilon;>"})
#   for i in N:
#     B = blocks[i]
#     insideB = range(1,len(B))
#     if len(insideB) == 1:
#       names = add_names(blocks[i][1], [[letters[i],""],[letters[i]+"'",""]], names)
#     if len(insideB) > 1:
#       for j in insideB:
#         # names = add_names(blocks[i][j], [letters[i]+'_'+str(j),letters[i]+'_'+str(j)+"'"],names)
#         names = add_names(blocks[i][j],[[letters[i],str(j)],[letters[i]+"'",str(j)]] ,names)
#          #[f'<{letters[i]}<SUB>{str(j)}</SUB>>',f"<{letters[i]}'<SUB>{str(j)}</SUB>>"],names)
#   return names
from collections import deque
def build_strict_index(X, p):
    """
    Given:
      X : list of elements
      p : dict-of-dicts so that p[x][y] ∈ X
          and p[x][y] == y exactly when x <= y in your intended order
    Returns:
      f : dict mapping each x∈X to an integer so that
          if f[x] <= f[y] then either p[x][y] == y
          or both f[x] < f[p[x][y]] and f[y] < f[p[x][y]].
    """
    # 1) build DAG edges for the strict order x ≺ y
    succ  = {x:set() for x in X}
    indeg = {x:0       for x in X}
    for x in X:
        for y in X:
            if x != y and p[x][y] == y:
                succ[x].add(y)
                indeg[y] += 1

    # 2) Kahn's algorithm
    q    = deque(x for x in X if indeg[x] == 0)
    topo = []
    while q:
        u = q.popleft()
        topo.append(u)
        for v in succ[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)

    if len(topo) != len(X):
        raise ValueError("Cycle detected in relation p[x][y]=y!")

    # 3) f(x) = position in topo order
    return { x:i for i,x in enumerate(topo) }


def fibers(model):
  blocks = Blocks(model)
  let_dic = build_strict_index([p[0][0] for p in blocks],model.operations["+"])
  n = len(blocks)
  N = range(n)
  SL = SLat(model)
  # letters = block_letters(n)
  if len(blocks[0]) == 1:
    lets = ["none"] + block_letters(n-1)
  else:
    lets = block_letters(n)
  shift = 0
  letters = []
  for i in N:
    B = blocks[i][0]
    lab = lets[let_dic[B[0]] + shift]
    # if B[0] == B[1]:
    #   shift = -1
    #   lab = "FP"
    letters += [lab]
  names = {0 : ['0',""], 1 : ['1',""]}
  for i in N[1:]:
    names = add_names(blocks[i][0], [["0",letters[i]],["1",letters[i]]], names)
    if blocks[i][0][0] == blocks[i][0][1]:
      names.update({blocks[i][0][0]: ["epsilon",""]})#"<&epsilon;>"})
  for i in N:
    B = blocks[i]
    insideB = range(1,len(B))
    if len(insideB) == 1:
      names = add_names(blocks[i][1], [[letters[i],""],[letters[i]+"'",""]], names)
    if len(insideB) > 1:
      for j in insideB:
        # names = add_names(blocks[i][j], [letters[i]+'_'+str(j),letters[i]+'_'+str(j)+"'"],names)
        names = add_names(blocks[i][j],[[letters[i],str(j)],[letters[i]+"'",str(j)]] ,names)
         #[f'<{letters[i]}<SUB>{str(j)}</SUB>>',f"<{letters[i]}'<SUB>{str(j)}</SUB>>"],names)
  return names

def name_format(names, format = "html"):
  out = []
  for i in range(len(names)):
    let = names[i][0]
    sub = names[i][1]
    if let == "epsilon": let = "&"+let+";" if format == "html" else r'\epsilon'
    if not sub == "": sub = "<sub>"+sub+"</sub>" if format == "html" else "_{"+sub+"}"
    n = f"<{let}{sub}>" if format == "html" else let + sub
    out.append(n)
  return out



# block_dict(M[0])



In [ ]:
def nh(m):
  return name_format(fibers(m))
def nt(m):
  return name_format(fibers(m),"tex")

def kl_po_props(model):
  klpo = ["x ^ x = x","x ^ (y ^ z) = (x ^ y) ^ z", "x ^ y = y ^ x", "x ^ y = x <-> x <= y"]
  def mc_is_kl(A):
    return prover9(A.diagram("")+ klpo,["x ^ (y' ^ z')' = ((x ^ y)' ^ (x ^ z)')'"],1000,0,A.cardinality,one = True) == []

  def mc_meet_is_si(A):
    B = prover9(A.diagram("")+ klpo,[],1000,0,A.cardinality,one = True)
    return isSI(B[0])
  if mc_is_kl(model):
    return ""
  else:
    st = r"$\begin{array}{c}" + "\n" + r"\leq \text{ is not a KL} "
    if mc_meet_is_si(model):
      st +=  r"\\ \hline" + "\n" + r"\text{with $\land$ is s.i.}" + "\n"
    st += r"\end{array}$" + "\n"
  # kl = "T" if mc_is_kl(A) else "F"
  # si = "T" if mc_meet_is_si(A) else "F"
  # st = r"$\begin{array}{c|c}" + "\n" + r"\leq \text{ is a KL}& " + kl + r"\\ \hline" + "\n"
  # st += r"\land \text{ expansion is s.i.}& " + si + " \n" + r"\end{array}$"
  return st


In [ ]:
def sort_fib(model):
  X = [x[0][0] for x in Blocks(model)]
  p = model.operations["+"]
  return sorted(Blocks(model), key=lambda x: build_strict_index(X,p)[x[0][0]])
# X = [p[0][0] for p in Blocks(A)]
# print(Blocks(A))
# print(X)
# print(sort_fib(A))
# print(nt(A))
# t = tex_table(t,elements=nt(A))
# f = sort_fib(A)

def fib_sets(model):
  f =  sort_fib(model)
  out = []
  for b in f:
    b_n = []
    for p in b:
      b_n += p if not(p[0] == p[1]) else [p[0]]
      # b_n += list(set(p))
    out += [b_n]
  return out

def block_indexed(blocks):
  f =  blocks
  out = []
  for b in f:
    out += b
  return out

def fib_sets_indexed(model):
  f =  fib_sets(model)
  return block_indexed(f)

# def fibers_t(model):
#   f =  sort_fib(model)
  # out = []
  # for b in f:
  #   out += b
  # return out

# print("Blocks:",Blocks(A))

# print("fibers_sets:",fib_sets(A))
# print("fibers_sets_index:",fib_sets_indexed(A))
# print("fibers_t:",fibers_t(A))
# print(nt(A))

def reorder_list(ls,permutation):
  return [ls[i] for i in permutation]

def fiber_table(model,op = "*"):
  f_ind = fib_sets_indexed(model)
  op = model.operations[op]
  Rows = [reorder_list(op[i],f_ind) for i in range(len(op))]
  return [reorder_list(Rows[i],f_ind) for i in range(len(op))]

# print(t)
# print(fiber_table(A))

def new_dict(model, dic):
  f_ind = fib_sets_indexed(model)
  # return {i:dic(model)[f_ind[i]] for i in range(model.cardinality)}
  return [dic(model)[f_ind[i]] for i in range(model.cardinality)]
# new_dict(A,nt)
# table_n = tex_table(fiber_table(A,"*"),"*" ,elements=new_dict(A,nt))



In [ ]:


def flag_hdash(blocks):
  flag = []
  for b in blocks:
    for i in range(len(b)):
      flag += [True] if i+1 == len(b) else [False]
  return flag



def block_table(blocks,op_arr):
  f_ind = block_indexed(blocks)
  op = op_arr
  if type(op[0]) == int:
    return reorder_list(op,f_ind)
  else:
    Rows = [reorder_list(op[i],f_ind) for i in range(len(op))]
    return reorder_list(Rows,f_ind)

def tex_table_n(arr,op = "*",elements = [],blocks = [],tex = False,disp = True):
    if elements == []:
      elements = range(0,len(arr))
    if op == "v":
      op = "\lor "
    if op == "^":
      op = "\land "
    if op == "\ ":
      op = r"\backslash "
    if op == "\\":
      op = r"\backslash "
    if op == "/ ":
      op = "\slash "
    if op == "//":
      op = "\slash "
    if op == "*":
      op = "\cdot "

    arr = block_table(blocks,arr)
    if type(arr[0]) == int:
      dim = 0
      arr = unary_tab(arr)
    elif type(arr[0]) == list:
      dim = len(arr[0])
    if blocks == []:
      blocks = [e for e in elements]
    block_ind = block_indexed(blocks)
    names = [elements[i] for i in block_ind]
    flag =  flag_hdash(blocks)[:-1]+[False]

    # Create the LaTeX table header
    latex_str = r"\begin{array}{c|"
    for i,b in enumerate(blocks):
      latex_str += "c"*len(b) if i+1==len(blocks) else "c"*len(b) + ":"
    latex_str +=  "}\n"
    # latex_str = r"\begin{array}{c|" + "c:" * dim + "}\n"

    # Add the first row for the column headers (operands)
    # header_row = op + " & " + " & ".join(map(str, elements)) + r" \\\hline" + "\n"
    header_row = op + " & " + " & ".join(names) + r" \\\hline" + "\n"
    latex_str += header_row

    # Fill the table rows
    for i, row in enumerate(arr):
        Row = [elements[j] for j in row]
        # latex_row = str(elements[i]) + " & " + " & ".join(map(str, Row)) + r" \\" + "\n"
        # latex_row = names[i] + " & " + " & ".join(Row) + r" \\\hdashline" + "\n"
        latex_row = names[i] + " & " + " & ".join(Row)
        if flag[i]:
          latex_row += r" \\ \hdashline" + "\n"
        else:
          latex_row += r" \\" + "\n"

        latex_str += latex_row

    # End the LaTeX array
    latex_str += r"\end{array}"

    # Display the LaTeX table
    if disp:
      display(Math(latex_str))
    if tex == True:
      print(f"LaTeX code:\n{latex_str}")
      print(" ")
    return latex_str

<>:23: SyntaxWarning: invalid escape sequence '\l'
<>:25: SyntaxWarning: invalid escape sequence '\l'
<>:26: SyntaxWarning: invalid escape sequence '\ '
<>:31: SyntaxWarning: invalid escape sequence '\s'
<>:33: SyntaxWarning: invalid escape sequence '\s'
<>:35: SyntaxWarning: invalid escape sequence '\c'
<>:23: SyntaxWarning: invalid escape sequence '\l'
<>:25: SyntaxWarning: invalid escape sequence '\l'
<>:26: SyntaxWarning: invalid escape sequence '\ '
<>:31: SyntaxWarning: invalid escape sequence '\s'
<>:33: SyntaxWarning: invalid escape sequence '\s'
<>:35: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_258/3606160868.py:23: SyntaxWarning: invalid escape sequence '\l'
  op = "\lor "
/tmp/ipykernel_258/3606160868.py:25: SyntaxWarning: invalid escape sequence '\l'
  op = "\land "
/tmp/ipykernel_258/3606160868.py:26: SyntaxWarning: invalid escape sequence '\ '
  if op == "\ ":
/tmp/ipykernel_258/3606160868.py:31: SyntaxWarning: invalid escape sequence '\s'
  op = "\slash "

In [ ]:
def add_ops(models,eqs):
  def term_def_ops(model_cl,eqs):
    if model_cl == []:
      return []
    else:
      return [prover9(model.diagram("") + eqs,[],1000,0,model.cardinality,one = True)[0] for model in model_cl]
  if type(models) == Model:
    models = [models]
    #return term_def_ops(models,eqs)
  if type(models) == str:
    models = []
  if type(models) == list and (models == [] or type(models[0]) == Model):
    return term_def_ops(models,eqs)
  elif type(models) == list and type(models[0]) == list and (models[0] == [] or type(models[0][0]) == Model):
    out = [[],[1]]
    for i in range(2,len(models)):
      print(i)
      out += [add_ops(models[i],eqs)]
    return out



def term_def_ops(model_cl,eqs):
    if model_cl == [] or type(model_cl) == str:
      return []
    else:
      return [prover9(model.diagram("") + eqs,[],1000,0,model.cardinality,one = True)[0] for model in model_cl]
# term_def_ops(McC[13],["x + y = (x'*y')'","x <= y <-> x*y = x  & y + x = y"])


In [ ]:
def add_opst(models,eqs,cnt_model = []):
  def term_def_ops(model_cl,eqs,cnt=[]):
    tout=[]
    if model_cl == []:
      return []
    else:
      for model in model_cl:
        a = p9(model.diagram("") + eqs,cnt,1000,0,model.cardinality)
        if a != [] and type(a[0]) != str:
          tout += a
      return tout
      # return [prover9(model.diagram("") + eqs,[],1000,0,model.cardinality,one = True)[0] for model in model_cl]
  if type(models) == Model:
    models = [models]
    #return term_def_ops(models,eqs)
  if type(models) == str:
    models = []
  if type(models) == list and (models == [] or type(models[0]) == Model):
    return term_def_ops(models,eqs, cnt_model)
  elif type(models) == list and type(models[0]) == list and (models[0] == [] or type(models[0][0]) == Model):
    out = [[],[1]]
    for i in range(2,len(models)):
      print(i)
      out += [add_opst(models[i],eqs,cnt_model)]
    return out

In [ ]:
def bool_block(gen = 1, letter = "a", in_num = 2):
  def elems(bin):
    return [b + [0] for b in bin] + [b + [1] for b in bin]
  def elmstr(v):
    return "".join(str(i) for i in v)
  B = [[]]
  for i in range(gen):
    B = elems(B)
  ls = []
  if in_num:
    for i in range(len(B)):
      ls.append(f"{letter}{elmstr(B[i])} = {str(i+in_num)}")
    # if in_num:
  # for i in range(len(B)):
  #   ls.append(f"{letter}{elmstr(B[i])} = {str(i+in_num)}")
  for b in B:
    ls.append(f"{letter}{elmstr(b)} * 0 = {letter}" + str(0)*gen)
    ls.append(f"{letter}{elmstr(b)}' = {letter}{elmstr([1 - i for i in b])}")
    for c in B:
      ls.append(f"{letter}{elmstr(b)} * {letter}{elmstr(c)} = {letter}{elmstr([i and j for i,j in zip(b,c)])}")
  return ls
letters(7)
S = [2,2,3,3,3,3,4]
Fax = []
cnt = 2
for i in range(len(S)):
  Fax += bool_block(S[i],letters(len(S))[i],cnt)
  cnt += 2**S[i]
Fax += ["a11 * b11 = g1111",

        "a10 * b00 = c000",
        "a10 * b10 = c100",
        "a10 * b01 = c010",

        "a01 * b00 = d000",
        "a01 * b10 = d100",
        "a01 * b01 = d010",

        "b10 * a00 = e000",
        "b01 * a00 = f000",
        "c111 * d111 = g1111",
        "c111 * e111 = g1111",
        "c111 * f111 = g1111",
        "d111 * e111 = g1111",
        "d111 * f111 = g1111",
        "e111 * f111 = g1111",
        ]
bool_block(3)



['a000 = 2',
 'a100 = 3',
 'a010 = 4',
 'a110 = 5',
 'a001 = 6',
 'a101 = 7',
 'a011 = 8',
 'a111 = 9',
 'a000 * 0 = a000',
 "a000' = a111",
 'a000 * a000 = a000',
 'a000 * a100 = a000',
 'a000 * a010 = a000',
 'a000 * a110 = a000',
 'a000 * a001 = a000',
 'a000 * a101 = a000',
 'a000 * a011 = a000',
 'a000 * a111 = a000',
 'a100 * 0 = a000',
 "a100' = a011",
 'a100 * a000 = a000',
 'a100 * a100 = a100',
 'a100 * a010 = a000',
 'a100 * a110 = a100',
 'a100 * a001 = a000',
 'a100 * a101 = a100',
 'a100 * a011 = a000',
 'a100 * a111 = a100',
 'a010 * 0 = a000',
 "a010' = a101",
 'a010 * a000 = a000',
 'a010 * a100 = a000',
 'a010 * a010 = a010',
 'a010 * a110 = a010',
 'a010 * a001 = a000',
 'a010 * a101 = a000',
 'a010 * a011 = a010',
 'a010 * a111 = a010',
 'a110 * 0 = a000',
 "a110' = a001",
 'a110 * a000 = a000',
 'a110 * a100 = a100',
 'a110 * a010 = a010',
 'a110 * a110 = a110',
 'a110 * a001 = a000',
 'a110 * a101 = a100',
 'a110 * a011 = a010',
 'a110 * a111 = a110',
 'a001 * 0 =